<a href="https://colab.research.google.com/github/RizalHabibi/-Bodyweight-Exercise-Classification-Using-CNN-LSTM/blob/main/Skripsi(Ekstraksi%2CTraining).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kode Ekstraksi

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp

# KONFIGURASI PATH DATA DI DRIVE
# 1. Path Input UCF101
PATH_UCF_TRAIN = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/UCF101/train'
PATH_UCF_TEST  = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/UCF101/test'

# 2. Path Input HMDB51 (Video & Split)
PATH_HMDB_VIDEO_ROOT = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/HMDB51/Video'
PATH_HMDB_SPLIT_DIR  = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/HMDB51/Split'

# 3. Path Output (45 Fitur)
PATH_OUTPUT_ROOT = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Dataset_NPY_Final_45Fitur'

#INISIALISASI MEDIAPIPE POSE
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, model_complexity=1)

# Menghitung sudut antara tiga titik (a, b, c) dengan b sebagai titik pusat
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b; bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

#Ekstraksi Fitur (36 -> 45)
def process_frame_to_45_features(landmarks):
    # Indeks 12 landmark utama: 11-12(bahu), 13-14(siku), 15-16(pergelangan tangan), 23-24(pinggul), 25-26(lutut), 27-28(pergelangan kaki)
    indices = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
    frame_raw = []

    xs = []
    ys = []

    for idx in indices:
        lm = landmarks[idx]
        frame_raw.extend([lm.x, lm.y, lm.z])
        xs.append(lm.x)
        ys.append(lm.y)

    frame_raw = np.array(frame_raw) # 36 Fitur

    def get_pt(idx): return np.array([frame_raw[idx], frame_raw[idx+1]])

    # Koordinat titik-titik kunci
    l_sh, r_sh = get_pt(0), get_pt(3)
    l_hip, r_hip = get_pt(18), get_pt(21)
    l_knee, r_knee = get_pt(24), get_pt(27)
    l_ank, r_ank = get_pt(30), get_pt(33)

    # Fitur geometris
    ankle_dist = np.abs(frame_raw[30] - frame_raw[33]) # Fitur 37, Jarak Horizontal Pergelangan Kaki
    knee_ver_diff = np.abs(frame_raw[25] - frame_raw[28]) # Fitur 38, Selisih Vertikal Lutut

    # Sudut (Fitur 39-42) - Normalized
    ang_l_knee = calculate_angle(l_hip, l_knee, l_ank)
    ang_r_knee = calculate_angle(r_hip, r_knee, r_ank)
    ang_l_hip = calculate_angle(l_sh, l_hip, l_knee)
    ang_r_hip = calculate_angle(r_sh, r_hip, r_knee)

    # Sudut Batang Tubuh Terhadap Vertikal - Fitur 43
    # Solusi untuk: Situp vs Pushup
    mid_shoulder = (l_sh + r_sh) / 2
    mid_hip = (l_hip + r_hip) / 2
    trunk_vec = mid_shoulder - mid_hip
    vertical_vec = np.array([0, -1])

    dot_product = np.dot(trunk_vec, vertical_vec)
    norm_trunk = np.linalg.norm(trunk_vec)
    if norm_trunk == 0: norm_trunk = 1.0

    # Hitung sudut (0=Tegak, 90=Tidur)
    trunk_angle_rad = np.arccos(np.clip(dot_product / norm_trunk, -1.0, 1.0))
    feat_trunk = np.degrees(trunk_angle_rad) / 180.0 # Normalisasi

    # KNEE SYMMETRY (Simetri Lutut) - Fitur 44
    # Solusi untuk: Squat (Simetris) vs Lunges (Asimetris)
    feat_knee_sym = np.abs(ang_l_knee - ang_r_knee) / 180.0

    # ASPECT RATIO (Rasio Dimensi Tubuh) - Fitur 45
    # Solusi untuk: Berdiri vs Tiduran
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    width = max_x - min_x
    height = max_y - min_y
    if width == 0: width = 0.001

    aspect_ratio = height / width
    # Clip di 3.0 dan normalisasi (biar range 0-1)
    feat_ratio = min(aspect_ratio, 3.0) / 3.0

    # Normalisasi sudut lama
    norm_angles = [ang_l_knee/180.0, ang_r_knee/180.0, ang_l_hip/180.0, ang_r_hip/180.0]

    # GABUNG SEMUA (45 Fitur)
    final_features = np.concatenate([
        frame_raw,           # 0-35 (36 fitur)
        [ankle_dist],        # 36
        [knee_ver_diff],     # 37
        norm_angles,         # 38-41 (4 fitur)
        [feat_trunk],        # 42 (NEW)
        [feat_knee_sym],     # 43 (NEW)
        [feat_ratio]         # 44 (NEW)
    ])

    return final_features

# Proses Satu Video
def process_single_video(video_path, save_path):
    if os.path.exists(save_path): return

    cap = cv2.VideoCapture(video_path)
    video_features = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        if results.pose_landmarks:
            # PANGGIL FUNGSI BARU (45 FITUR)
            feats = process_frame_to_45_features(results.pose_landmarks.landmark)
            video_features.append(feats)
    cap.release()

    if len(video_features) > 0:
        np.save(save_path, np.array(video_features))
        print(f"OK: {os.path.basename(video_path)} ({len(video_features)} frames)")
    else:
        print(f"Gagal/Kosong: {os.path.basename(video_path)}")


#PROSES UCF101
def process_ucf_folder(source_root, split_name):
    print(f"\n MEMPROSES UCF101 - {split_name.upper()}...")
    if not os.path.exists(source_root):
        print(f" Folder UCF {split_name} tidak ditemukan!")
        return

    for class_name in os.listdir(source_root):
        class_path = os.path.join(source_root, class_name)
        if not os.path.isdir(class_path): continue

        target_dir = os.path.join(PATH_OUTPUT_ROOT, split_name, class_name)
        os.makedirs(target_dir, exist_ok=True)

        print(f" Kelas: {class_name}")
        for vid in os.listdir(class_path):
            if vid.endswith(('.avi', '.mp4')):
                src = os.path.join(class_path, vid)
                dst = os.path.join(target_dir, vid.replace('.avi','.npy').replace('.mp4','.npy'))
                process_single_video(src, dst)


#PROSES HMDB51
def process_hmdb_situp():
    print(f"\n MEMPROSES HMDB51 - SITUP...")

    split_file_path = os.path.join(PATH_HMDB_SPLIT_DIR, 'situp_test_split1.txt')
    if not os.path.exists(split_file_path):
        candidates = [f for f in os.listdir(PATH_HMDB_SPLIT_DIR) if 'situp' in f and 'split1' in f]
        if candidates:
            split_file_path = os.path.join(PATH_HMDB_SPLIT_DIR, candidates[0])
        else:
            print(" File Split HMDB Situp tidak ditemukan!")
            return

    print(f"Menggunakan Split File: {os.path.basename(split_file_path)}")

    with open(split_file_path, 'r') as f:
        lines = f.readlines()

    path_situp_video = os.path.join(PATH_HMDB_VIDEO_ROOT, 'situp')
    if not os.path.exists(path_situp_video):
        path_situp_video = PATH_HMDB_VIDEO_ROOT

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2: continue

        vid_name = parts[0]
        split_id = parts[1]

        split_folder = ""
        if split_id == '1': split_folder = 'train'
        elif split_id == '2': split_folder = 'test'
        else: continue

        src = os.path.join(path_situp_video, vid_name)
        target_dir = os.path.join(PATH_OUTPUT_ROOT, split_folder, 'Situp')
        os.makedirs(target_dir, exist_ok=True)
        dst = os.path.join(target_dir, vid_name.replace('.avi','.npy').replace('.mp4','.npy'))

        if os.path.exists(src):
            process_single_video(src, dst)


#EKSEKUSI
if __name__ == "__main__":
    process_ucf_folder(PATH_UCF_TRAIN, 'train')
    process_ucf_folder(PATH_UCF_TEST, 'test')
    process_hmdb_situp()

    print("\n SELESAI! Dataset NPY Final (45 Fitur) siap di:")
    print(PATH_OUTPUT_ROOT)

In [ ]:

🎥 MEMPROSES UCF101 - TRAIN...
   📂 Kelas: jumpingjack
   ✅ OK: v_JumpingJack_g08_c01.avi (80 frames)
   ✅ OK: v_JumpingJack_g08_c02.avi (83 frames)
   ✅ OK: v_JumpingJack_g08_c03.avi (81 frames)
   ✅ OK: v_JumpingJack_g08_c04.avi (80 frames)
   ✅ OK: v_JumpingJack_g09_c01.avi (93 frames)
   ✅ OK: v_JumpingJack_g09_c02.avi (92 frames)
   ✅ OK: v_JumpingJack_g09_c03.avi (89 frames)
   ✅ OK: v_JumpingJack_g09_c04.avi (89 frames)
   ✅ OK: v_JumpingJack_g09_c05.avi (90 frames)
   ✅ OK: v_JumpingJack_g09_c06.avi (90 frames)
   ✅ OK: v_JumpingJack_g09_c07.avi (92 frames)
   ✅ OK: v_JumpingJack_g10_c01.avi (68 frames)
   ✅ OK: v_JumpingJack_g10_c02.avi (68 frames)
   ✅ OK: v_JumpingJack_g10_c03.avi (66 frames)
   ✅ OK: v_JumpingJack_g10_c04.avi (60 frames)
   ✅ OK: v_JumpingJack_g10_c05.avi (64 frames)
   ✅ OK: v_JumpingJack_g10_c06.avi (66 frames)
   ✅ OK: v_JumpingJack_g11_c01.avi (39 frames)
   ✅ OK: v_JumpingJack_g11_c02.avi (76 frames)
   ✅ OK: v_JumpingJack_g11_c03.avi (67 frames)
   ✅ OK: v_JumpingJack_g12_c01.avi (95 frames)
   ✅ OK: v_JumpingJack_g11_c04.avi (80 frames)
   ✅ OK: v_JumpingJack_g12_c02.avi (95 frames)
   ✅ OK: v_JumpingJack_g12_c03.avi (96 frames)
   ✅ OK: v_JumpingJack_g12_c04.avi (96 frames)
   ✅ OK: v_JumpingJack_g13_c01.avi (46 frames)
   ✅ OK: v_JumpingJack_g13_c02.avi (110 frames)
   ✅ OK: v_JumpingJack_g13_c03.avi (97 frames)
   ✅ OK: v_JumpingJack_g13_c04.avi (92 frames)
   ✅ OK: v_JumpingJack_g13_c05.avi (95 frames)
   ✅ OK: v_JumpingJack_g13_c06.avi (80 frames)
   ✅ OK: v_JumpingJack_g13_c07.avi (90 frames)
   ✅ OK: v_JumpingJack_g14_c01.avi (96 frames)
   ✅ OK: v_JumpingJack_g14_c02.avi (60 frames)
   ✅ OK: v_JumpingJack_g14_c03.avi (58 frames)
   ✅ OK: v_JumpingJack_g14_c04.avi (60 frames)
   ✅ OK: v_JumpingJack_g15_c01.avi (82 frames)
   ✅ OK: v_JumpingJack_g15_c02.avi (70 frames)
   ✅ OK: v_JumpingJack_g15_c03.avi (82 frames)
   ✅ OK: v_JumpingJack_g15_c04.avi (82 frames)
   ✅ OK: v_JumpingJack_g16_c01.avi (78 frames)
   ✅ OK: v_JumpingJack_g16_c02.avi (69 frames)
   ✅ OK: v_JumpingJack_g16_c03.avi (66 frames)
   ✅ OK: v_JumpingJack_g16_c04.avi (65 frames)
   ✅ OK: v_JumpingJack_g17_c01.avi (93 frames)
   ✅ OK: v_JumpingJack_g17_c02.avi (94 frames)
   ✅ OK: v_JumpingJack_g17_c03.avi (101 frames)
   ✅ OK: v_JumpingJack_g17_c04.avi (94 frames)
   ✅ OK: v_JumpingJack_g18_c01.avi (92 frames)
   ✅ OK: v_JumpingJack_g18_c02.avi (70 frames)
   ✅ OK: v_JumpingJack_g18_c03.avi (94 frames)
   ✅ OK: v_JumpingJack_g18_c04.avi (97 frames)
   ✅ OK: v_JumpingJack_g19_c01.avi (83 frames)
   ✅ OK: v_JumpingJack_g19_c02.avi (82 frames)
   ✅ OK: v_JumpingJack_g19_c03.avi (84 frames)
   ✅ OK: v_JumpingJack_g19_c04.avi (77 frames)
   ✅ OK: v_JumpingJack_g19_c05.avi (79 frames)
   ✅ OK: v_JumpingJack_g19_c06.avi (78 frames)
   ✅ OK: v_JumpingJack_g19_c07.avi (84 frames)
   ✅ OK: v_JumpingJack_g20_c01.avi (116 frames)
   ✅ OK: v_JumpingJack_g20_c02.avi (112 frames)
   ✅ OK: v_JumpingJack_g20_c03.avi (138 frames)
   ✅ OK: v_JumpingJack_g20_c04.avi (94 frames)
   ✅ OK: v_JumpingJack_g21_c01.avi (93 frames)
   ✅ OK: v_JumpingJack_g21_c02.avi (93 frames)
   ✅ OK: v_JumpingJack_g21_c04.avi (128 frames)
   ✅ OK: v_JumpingJack_g21_c03.avi (93 frames)
   ✅ OK: v_JumpingJack_g22_c01.avi (89 frames)
   ✅ OK: v_JumpingJack_g22_c02.avi (60 frames)
   ✅ OK: v_JumpingJack_g22_c04.avi (57 frames)
   ✅ OK: v_JumpingJack_g22_c03.avi (84 frames)
   ✅ OK: v_JumpingJack_g23_c01.avi (85 frames)
   ✅ OK: v_JumpingJack_g23_c02.avi (85 frames)
   ✅ OK: v_JumpingJack_g23_c03.avi (79 frames)
   ✅ OK: v_JumpingJack_g23_c04.avi (85 frames)
   ✅ OK: v_JumpingJack_g24_c01.avi (92 frames)
   ✅ OK: v_JumpingJack_g24_c02.avi (91 frames)
   ✅ OK: v_JumpingJack_g24_c03.avi (93 frames)
   ✅ OK: v_JumpingJack_g24_c04.avi (88 frames)
   ✅ OK: v_JumpingJack_g25_c01.avi (86 frames)
   ✅ OK: v_JumpingJack_g25_c02.avi (88 frames)
   ✅ OK: v_JumpingJack_g25_c03.avi (87 frames)
   ✅ OK: v_JumpingJack_g25_c04.avi (88 frames)
   ✅ OK: v_JumpingJack_g25_c05.avi (86 frames)
   ✅ OK: v_JumpingJack_g25_c06.avi (87 frames)
   ✅ OK: v_JumpingJack_g25_c07.avi (88 frames)
   📂 Kelas: lunges
   ✅ OK: v_Lunges_g08_c01.avi (159 frames)
   ✅ OK: v_Lunges_g08_c02.avi (171 frames)
   ✅ OK: v_Lunges_g08_c03.avi (175 frames)
   ✅ OK: v_Lunges_g08_c04.avi (161 frames)
   ✅ OK: v_Lunges_g09_c01.avi (176 frames)
   ✅ OK: v_Lunges_g09_c02.avi (201 frames)
   ✅ OK: v_Lunges_g09_c03.avi (205 frames)
   ✅ OK: v_Lunges_g09_c04.avi (191 frames)
   ✅ OK: v_Lunges_g10_c01.avi (255 frames)
   ✅ OK: v_Lunges_g10_c02.avi (262 frames)
   ✅ OK: v_Lunges_g10_c03.avi (251 frames)
   ✅ OK: v_Lunges_g10_c04.avi (197 frames)
   ✅ OK: v_Lunges_g10_c05.avi (244 frames)
   ✅ OK: v_Lunges_g10_c06.avi (263 frames)
   ✅ OK: v_Lunges_g11_c01.avi (241 frames)
   ✅ OK: v_Lunges_g11_c02.avi (241 frames)
   ✅ OK: v_Lunges_g11_c03.avi (256 frames)
   ✅ OK: v_Lunges_g11_c04.avi (244 frames)
   ✅ OK: v_Lunges_g11_c05.avi (229 frames)
   ✅ OK: v_Lunges_g11_c06.avi (259 frames)
   ✅ OK: v_Lunges_g11_c07.avi (254 frames)
   ✅ OK: v_Lunges_g12_c01.avi (252 frames)
   ✅ OK: v_Lunges_g12_c02.avi (263 frames)
   ✅ OK: v_Lunges_g12_c03.avi (253 frames)
   ✅ OK: v_Lunges_g12_c04.avi (252 frames)
   ✅ OK: v_Lunges_g12_c05.avi (245 frames)
   ✅ OK: v_Lunges_g13_c01.avi (266 frames)
   ✅ OK: v_Lunges_g13_c02.avi (249 frames)
   ✅ OK: v_Lunges_g13_c03.avi (224 frames)
   ✅ OK: v_Lunges_g13_c04.avi (241 frames)
   ✅ OK: v_Lunges_g13_c05.avi (157 frames)
   ✅ OK: v_Lunges_g13_c06.avi (129 frames)
   ✅ OK: v_Lunges_g14_c01.avi (255 frames)
   ✅ OK: v_Lunges_g14_c02.avi (263 frames)
   ✅ OK: v_Lunges_g14_c03.avi (262 frames)
   ✅ OK: v_Lunges_g14_c04.avi (249 frames)
   ✅ OK: v_Lunges_g14_c05.avi (241 frames)
   ✅ OK: v_Lunges_g14_c06.avi (255 frames)
   ✅ OK: v_Lunges_g14_c07.avi (242 frames)
   ✅ OK: v_Lunges_g15_c01.avi (117 frames)
   ✅ OK: v_Lunges_g15_c02.avi (97 frames)
   ✅ OK: v_Lunges_g15_c03.avi (106 frames)
   ✅ OK: v_Lunges_g15_c04.avi (86 frames)
   ✅ OK: v_Lunges_g16_c01.avi (254 frames)
   ✅ OK: v_Lunges_g16_c02.avi (244 frames)
   ✅ OK: v_Lunges_g16_c03.avi (238 frames)
   ✅ OK: v_Lunges_g16_c04.avi (248 frames)
   ✅ OK: v_Lunges_g17_c01.avi (242 frames)
   ✅ OK: v_Lunges_g17_c02.avi (255 frames)
   ✅ OK: v_Lunges_g17_c03.avi (254 frames)
   ✅ OK: v_Lunges_g17_c04.avi (252 frames)
   ✅ OK: v_Lunges_g18_c01.avi (256 frames)
   ✅ OK: v_Lunges_g18_c02.avi (249 frames)
   ✅ OK: v_Lunges_g18_c03.avi (253 frames)
   ✅ OK: v_Lunges_g18_c04.avi (246 frames)
   ✅ OK: v_Lunges_g18_c05.avi (254 frames)
   ✅ OK: v_Lunges_g19_c01.avi (118 frames)
   ✅ OK: v_Lunges_g19_c02.avi (247 frames)
   ✅ OK: v_Lunges_g19_c03.avi (248 frames)
   ✅ OK: v_Lunges_g19_c04.avi (257 frames)
   ✅ OK: v_Lunges_g19_c05.avi (259 frames)
   ✅ OK: v_Lunges_g19_c06.avi (239 frames)
   ✅ OK: v_Lunges_g19_c07.avi (241 frames)
   ✅ OK: v_Lunges_g20_c01.avi (48 frames)
   ✅ OK: v_Lunges_g20_c02.avi (47 frames)
   ✅ OK: v_Lunges_g20_c04.avi (48 frames)
   ✅ OK: v_Lunges_g20_c03.avi (47 frames)
   ✅ OK: v_Lunges_g21_c01.avi (267 frames)
   ✅ OK: v_Lunges_g21_c02.avi (241 frames)
   ✅ OK: v_Lunges_g21_c03.avi (259 frames)
   ✅ OK: v_Lunges_g21_c04.avi (253 frames)
   ✅ OK: v_Lunges_g21_c05.avi (236 frames)
   ✅ OK: v_Lunges_g22_c01.avi (190 frames)
   ✅ OK: v_Lunges_g22_c02.avi (204 frames)
   ✅ OK: v_Lunges_g22_c03.avi (203 frames)
   ✅ OK: v_Lunges_g22_c04.avi (208 frames)
   ✅ OK: v_Lunges_g23_c01.avi (119 frames)
   ✅ OK: v_Lunges_g23_c02.avi (151 frames)
   ✅ OK: v_Lunges_g23_c03.avi (180 frames)
   ✅ OK: v_Lunges_g23_c04.avi (202 frames)
   ✅ OK: v_Lunges_g23_c05.avi (247 frames)
   ✅ OK: v_Lunges_g23_c06.avi (248 frames)
   ✅ OK: v_Lunges_g24_c01.avi (192 frames)
   ✅ OK: v_Lunges_g24_c02.avi (193 frames)
   ✅ OK: v_Lunges_g24_c03.avi (193 frames)
   ✅ OK: v_Lunges_g24_c04.avi (176 frames)
   ✅ OK: v_Lunges_g25_c01.avi (235 frames)
   ✅ OK: v_Lunges_g25_c02.avi (231 frames)
   ✅ OK: v_Lunges_g25_c03.avi (245 frames)
   ✅ OK: v_Lunges_g25_c04.avi (251 frames)
   📂 Kelas: pushups
   ✅ OK: v_PushUps_g08_c01.avi (84 frames)
   ✅ OK: v_PushUps_g08_c02.avi (84 frames)
   ✅ OK: v_PushUps_g08_c03.avi (84 frames)
   ✅ OK: v_PushUps_g09_c01.avi (84 frames)
   ✅ OK: v_PushUps_g08_c04.avi (84 frames)
   ✅ OK: v_PushUps_g09_c02.avi (84 frames)
   ✅ OK: v_PushUps_g09_c03.avi (84 frames)
   ✅ OK: v_PushUps_g09_c04.avi (84 frames)
   ✅ OK: v_PushUps_g10_c01.avi (66 frames)
   ✅ OK: v_PushUps_g10_c02.avi (62 frames)
   ✅ OK: v_PushUps_g10_c03.avi (66 frames)
   ✅ OK: v_PushUps_g10_c04.avi (71 frames)
   ✅ OK: v_PushUps_g11_c01.avi (49 frames)
   ✅ OK: v_PushUps_g11_c02.avi (46 frames)
   ✅ OK: v_PushUps_g11_c03.avi (42 frames)
   ✅ OK: v_PushUps_g11_c04.avi (36 frames)
   ✅ OK: v_PushUps_g12_c01.avi (73 frames)
   ✅ OK: v_PushUps_g12_c02.avi (74 frames)
   ✅ OK: v_PushUps_g12_c03.avi (70 frames)
   ✅ OK: v_PushUps_g12_c04.avi (72 frames)
   ✅ OK: v_PushUps_g13_c01.avi (72 frames)
   ✅ OK: v_PushUps_g13_c02.avi (72 frames)
   ✅ OK: v_PushUps_g13_c03.avi (48 frames)
   ✅ OK: v_PushUps_g14_c01.avi (102 frames)
   ✅ OK: v_PushUps_g13_c04.avi (54 frames)
   ✅ OK: v_PushUps_g14_c02.avi (98 frames)
   ✅ OK: v_PushUps_g14_c03.avi (100 frames)
   ✅ OK: v_PushUps_g14_c04.avi (94 frames)
   ✅ OK: v_PushUps_g15_c01.avi (102 frames)
   ✅ OK: v_PushUps_g15_c02.avi (85 frames)
   ✅ OK: v_PushUps_g15_c03.avi (85 frames)
   ✅ OK: v_PushUps_g15_c04.avi (87 frames)
   ✅ OK: v_PushUps_g16_c01.avi (69 frames)
   ✅ OK: v_PushUps_g16_c02.avi (76 frames)
   ✅ OK: v_PushUps_g16_c03.avi (33 frames)
   ✅ OK: v_PushUps_g16_c04.avi (29 frames)
   ✅ OK: v_PushUps_g17_c01.avi (79 frames)
   ✅ OK: v_PushUps_g17_c02.avi (81 frames)
   ✅ OK: v_PushUps_g17_c03.avi (74 frames)
   ✅ OK: v_PushUps_g17_c04.avi (79 frames)
   ✅ OK: v_PushUps_g18_c01.avi (71 frames)
   ✅ OK: v_PushUps_g18_c03.avi (75 frames)
   ✅ OK: v_PushUps_g18_c02.avi (71 frames)
   ✅ OK: v_PushUps_g18_c04.avi (71 frames)
   ✅ OK: v_PushUps_g19_c01.avi (72 frames)
   ✅ OK: v_PushUps_g19_c02.avi (75 frames)
   ✅ OK: v_PushUps_g19_c03.avi (76 frames)
   ✅ OK: v_PushUps_g19_c04.avi (76 frames)
   ✅ OK: v_PushUps_g20_c01.avi (59 frames)
   ✅ OK: v_PushUps_g20_c02.avi (30 frames)
   ✅ OK: v_PushUps_g20_c03.avi (51 frames)
   ✅ OK: v_PushUps_g20_c04.avi (37 frames)
   ✅ OK: v_PushUps_g21_c01.avi (90 frames)
   ✅ OK: v_PushUps_g21_c02.avi (166 frames)
   ✅ OK: v_PushUps_g21_c03.avi (124 frames)
   ✅ OK: v_PushUps_g21_c04.avi (102 frames)
   ✅ OK: v_PushUps_g22_c01.avi (57 frames)
   ✅ OK: v_PushUps_g22_c02.avi (108 frames)
   ✅ OK: v_PushUps_g22_c03.avi (67 frames)
   ✅ OK: v_PushUps_g23_c01.avi (63 frames)
   ✅ OK: v_PushUps_g22_c04.avi (96 frames)
   ✅ OK: v_PushUps_g23_c02.avi (61 frames)
   ✅ OK: v_PushUps_g23_c03.avi (62 frames)
   ✅ OK: v_PushUps_g23_c04.avi (60 frames)
   ✅ OK: v_PushUps_g24_c01.avi (71 frames)
   ✅ OK: v_PushUps_g24_c02.avi (68 frames)
   ✅ OK: v_PushUps_g24_c03.avi (69 frames)
   ✅ OK: v_PushUps_g24_c04.avi (70 frames)
   ✅ OK: v_PushUps_g25_c01.avi (90 frames)
   ✅ OK: v_PushUps_g25_c03.avi (111 frames)
   ✅ OK: v_PushUps_g25_c02.avi (109 frames)
   ✅ OK: v_PushUps_g25_c04.avi (101 frames)
   📂 Kelas: squat
   ✅ OK: v_BodyWeightSquats_g08_c01.avi (76 frames)
   ✅ OK: v_BodyWeightSquats_g08_c02.avi (106 frames)
   ✅ OK: v_BodyWeightSquats_g08_c03.avi (49 frames)
   ✅ OK: v_BodyWeightSquats_g08_c04.avi (104 frames)
   ✅ OK: v_BodyWeightSquats_g09_c01.avi (200 frames)
   ✅ OK: v_BodyWeightSquats_g09_c03.avi (115 frames)
   ✅ OK: v_BodyWeightSquats_g09_c02.avi (200 frames)
   ✅ OK: v_BodyWeightSquats_g09_c04.avi (123 frames)
   ✅ OK: v_BodyWeightSquats_g09_c05.avi (226 frames)
   ✅ OK: v_BodyWeightSquats_g09_c06.avi (260 frames)
   ✅ OK: v_BodyWeightSquats_g09_c07.avi (233 frames)
   ✅ OK: v_BodyWeightSquats_g10_c01.avi (168 frames)
   ✅ OK: v_BodyWeightSquats_g10_c02.avi (160 frames)
   ✅ OK: v_BodyWeightSquats_g10_c03.avi (152 frames)
   ✅ OK: v_BodyWeightSquats_g10_c04.avi (148 frames)
   ✅ OK: v_BodyWeightSquats_g10_c05.avi (158 frames)
   ✅ OK: v_BodyWeightSquats_g11_c01.avi (81 frames)
   ✅ OK: v_BodyWeightSquats_g11_c02.avi (95 frames)
   ✅ OK: v_BodyWeightSquats_g11_c03.avi (86 frames)
   ✅ OK: v_BodyWeightSquats_g11_c04.avi (77 frames)
   ✅ OK: v_BodyWeightSquats_g12_c02.avi (143 frames)
   ✅ OK: v_BodyWeightSquats_g12_c01.avi (237 frames)
   ✅ OK: v_BodyWeightSquats_g12_c03.avi (120 frames)
   ✅ OK: v_BodyWeightSquats_g12_c04.avi (193 frames)
   ✅ OK: v_BodyWeightSquats_g13_c02.avi (62 frames)
   ✅ OK: v_BodyWeightSquats_g13_c01.avi (112 frames)
   ✅ OK: v_BodyWeightSquats_g13_c03.avi (115 frames)
   ✅ OK: v_BodyWeightSquats_g14_c01.avi (110 frames)
   ✅ OK: v_BodyWeightSquats_g13_c04.avi (75 frames)
   ✅ OK: v_BodyWeightSquats_g14_c02.avi (114 frames)
   ✅ OK: v_BodyWeightSquats_g14_c03.avi (114 frames)
   ✅ OK: v_BodyWeightSquats_g15_c01.avi (142 frames)
   ✅ OK: v_BodyWeightSquats_g14_c04.avi (111 frames)
   ✅ OK: v_BodyWeightSquats_g15_c02.avi (122 frames)
   ✅ OK: v_BodyWeightSquats_g15_c04.avi (126 frames)
   ✅ OK: v_BodyWeightSquats_g15_c03.avi (137 frames)
   ✅ OK: v_BodyWeightSquats_g16_c01.avi (74 frames)
   ✅ OK: v_BodyWeightSquats_g16_c02.avi (75 frames)
   ✅ OK: v_BodyWeightSquats_g16_c03.avi (81 frames)
   ✅ OK: v_BodyWeightSquats_g16_c04.avi (77 frames)
   ✅ OK: v_BodyWeightSquats_g17_c01.avi (179 frames)
   ✅ OK: v_BodyWeightSquats_g17_c02.avi (174 frames)
   ✅ OK: v_BodyWeightSquats_g17_c03.avi (170 frames)
   ✅ OK: v_BodyWeightSquats_g17_c04.avi (144 frames)
   ✅ OK: v_BodyWeightSquats_g18_c01.avi (203 frames)
   ✅ OK: v_BodyWeightSquats_g18_c02.avi (214 frames)
   ✅ OK: v_BodyWeightSquats_g18_c03.avi (211 frames)
   ✅ OK: v_BodyWeightSquats_g18_c04.avi (223 frames)
   ✅ OK: v_BodyWeightSquats_g19_c01.avi (135 frames)
   ✅ OK: v_BodyWeightSquats_g19_c02.avi (112 frames)
   ✅ OK: v_BodyWeightSquats_g19_c03.avi (144 frames)
   ✅ OK: v_BodyWeightSquats_g19_c04.avi (108 frames)
   ✅ OK: v_BodyWeightSquats_g20_c01.avi (99 frames)
   ✅ OK: v_BodyWeightSquats_g20_c02.avi (174 frames)
   ✅ OK: v_BodyWeightSquats_g20_c03.avi (94 frames)
   ✅ OK: v_BodyWeightSquats_g20_c04.avi (164 frames)
   ✅ OK: v_BodyWeightSquats_g20_c05.avi (135 frames)
   ✅ OK: v_BodyWeightSquats_g20_c06.avi (167 frames)
   ✅ OK: v_BodyWeightSquats_g21_c01.avi (51 frames)
   ✅ OK: v_BodyWeightSquats_g21_c03.avi (54 frames)
   ✅ OK: v_BodyWeightSquats_g21_c02.avi (64 frames)
   ✅ OK: v_BodyWeightSquats_g21_c04.avi (87 frames)
   ✅ OK: v_BodyWeightSquats_g22_c01.avi (90 frames)
   ✅ OK: v_BodyWeightSquats_g22_c02.avi (62 frames)
   ✅ OK: v_BodyWeightSquats_g22_c03.avi (138 frames)
   ✅ OK: v_BodyWeightSquats_g23_c01.avi (52 frames)
   ✅ OK: v_BodyWeightSquats_g22_c04.avi (62 frames)
   ✅ OK: v_BodyWeightSquats_g23_c02.avi (76 frames)
   ✅ OK: v_BodyWeightSquats_g23_c03.avi (73 frames)
   ✅ OK: v_BodyWeightSquats_g23_c04.avi (66 frames)
   ✅ OK: v_BodyWeightSquats_g24_c01.avi (229 frames)
   ✅ OK: v_BodyWeightSquats_g24_c02.avi (221 frames)
   ✅ OK: v_BodyWeightSquats_g24_c03.avi (236 frames)
   ✅ OK: v_BodyWeightSquats_g24_c05.avi (78 frames)
   ✅ OK: v_BodyWeightSquats_g24_c04.avi (237 frames)
   ✅ OK: v_BodyWeightSquats_g25_c02.avi (110 frames)
   ✅ OK: v_BodyWeightSquats_g25_c01.avi (103 frames)
   ✅ OK: v_BodyWeightSquats_g25_c03.avi (116 frames)
   ✅ OK: v_BodyWeightSquats_g25_c04.avi (108 frames)
   ✅ OK: v_BodyWeightSquats_g25_c06.avi (101 frames)
   ✅ OK: v_BodyWeightSquats_g25_c05.avi (106 frames)
   ✅ OK: v_BodyWeightSquats_g25_c07.avi (125 frames)

🎥 MEMPROSES UCF101 - TEST...
   📂 Kelas: jumpingjack
   ✅ OK: v_JumpingJack_g01_c01.avi (93 frames)
   ✅ OK: v_JumpingJack_g01_c02.avi (91 frames)
   ✅ OK: v_JumpingJack_g01_c03.avi (92 frames)
   ✅ OK: v_JumpingJack_g01_c04.avi (89 frames)
   ✅ OK: v_JumpingJack_g01_c05.avi (89 frames)
   ✅ OK: v_JumpingJack_g01_c06.avi (88 frames)
   ✅ OK: v_JumpingJack_g01_c07.avi (89 frames)
   ✅ OK: v_JumpingJack_g02_c01.avi (63 frames)
   ✅ OK: v_JumpingJack_g02_c02.avi (63 frames)
   ✅ OK: v_JumpingJack_g02_c03.avi (101 frames)
   ✅ OK: v_JumpingJack_g02_c04.avi (94 frames)
   ✅ OK: v_JumpingJack_g03_c01.avi (56 frames)
   ✅ OK: v_JumpingJack_g03_c02.avi (87 frames)
   ✅ OK: v_JumpingJack_g03_c03.avi (82 frames)
   ✅ OK: v_JumpingJack_g03_c04.avi (83 frames)
   ✅ OK: v_JumpingJack_g04_c01.avi (54 frames)
   ✅ OK: v_JumpingJack_g04_c02.avi (78 frames)
   ✅ OK: v_JumpingJack_g04_c03.avi (80 frames)
   ✅ OK: v_JumpingJack_g05_c01.avi (81 frames)
   ✅ OK: v_JumpingJack_g04_c04.avi (85 frames)
   ✅ OK: v_JumpingJack_g05_c02.avi (82 frames)
   ✅ OK: v_JumpingJack_g05_c03.avi (100 frames)
   ✅ OK: v_JumpingJack_g05_c05.avi (79 frames)
   ✅ OK: v_JumpingJack_g05_c04.avi (81 frames)
   ✅ OK: v_JumpingJack_g05_c06.avi (82 frames)
   ✅ OK: v_JumpingJack_g06_c01.avi (71 frames)
   ✅ OK: v_JumpingJack_g06_c02.avi (72 frames)
   ✅ OK: v_JumpingJack_g06_c03.avi (79 frames)
   ✅ OK: v_JumpingJack_g06_c04.avi (72 frames)
   ✅ OK: v_JumpingJack_g06_c05.avi (69 frames)
   ✅ OK: v_JumpingJack_g06_c06.avi (72 frames)
   ✅ OK: v_JumpingJack_g06_c07.avi (66 frames)
   ✅ OK: v_JumpingJack_g07_c01.avi (87 frames)
   ✅ OK: v_JumpingJack_g07_c02.avi (84 frames)
   ✅ OK: v_JumpingJack_g07_c03.avi (84 frames)
   ✅ OK: v_JumpingJack_g07_c04.avi (87 frames)
   ✅ OK: v_JumpingJack_g07_c05.avi (88 frames)
   📂 Kelas: lunges
   ✅ OK: v_Lunges_g01_c01.avi (242 frames)
   ✅ OK: v_Lunges_g01_c02.avi (55 frames)
   ✅ OK: v_Lunges_g01_c03.avi (144 frames)
   ✅ OK: v_Lunges_g01_c04.avi (262 frames)
   ✅ OK: v_Lunges_g01_c05.avi (193 frames)
   ✅ OK: v_Lunges_g01_c06.avi (139 frames)
   ✅ OK: v_Lunges_g01_c07.avi (168 frames)
   ✅ OK: v_Lunges_g02_c01.avi (93 frames)
   ✅ OK: v_Lunges_g02_c02.avi (110 frames)
   ✅ OK: v_Lunges_g02_c03.avi (114 frames)
   ✅ OK: v_Lunges_g02_c04.avi (133 frames)
   ✅ OK: v_Lunges_g03_c01.avi (238 frames)
   ✅ OK: v_Lunges_g03_c02.avi (182 frames)
   ✅ OK: v_Lunges_g03_c03.avi (197 frames)
   ✅ OK: v_Lunges_g03_c04.avi (236 frames)
   ✅ OK: v_Lunges_g04_c01.avi (262 frames)
   ✅ OK: v_Lunges_g04_c02.avi (260 frames)
   ✅ OK: v_Lunges_g04_c03.avi (248 frames)
   ✅ OK: v_Lunges_g04_c04.avi (184 frames)
   ✅ OK: v_Lunges_g05_c01.avi (234 frames)
   ✅ OK: v_Lunges_g05_c02.avi (252 frames)
   ✅ OK: v_Lunges_g05_c03.avi (217 frames)
   ✅ OK: v_Lunges_g05_c04.avi (185 frames)
   ✅ OK: v_Lunges_g06_c01.avi (253 frames)
   ✅ OK: v_Lunges_g06_c02.avi (122 frames)
   ✅ OK: v_Lunges_g06_c03.avi (256 frames)
   ✅ OK: v_Lunges_g06_c04.avi (266 frames)
   ✅ OK: v_Lunges_g06_c05.avi (255 frames)
   ✅ OK: v_Lunges_g06_c06.avi (102 frames)
   ✅ OK: v_Lunges_g06_c07.avi (262 frames)
   ✅ OK: v_Lunges_g07_c01.avi (248 frames)
   ✅ OK: v_Lunges_g07_c02.avi (246 frames)
   ✅ OK: v_Lunges_g07_c03.avi (246 frames)
   ✅ OK: v_Lunges_g07_c04.avi (265 frames)
   ✅ OK: v_Lunges_g07_c05.avi (238 frames)
   ✅ OK: v_Lunges_g07_c06.avi (251 frames)
   ✅ OK: v_Lunges_g07_c07.avi (254 frames)
   📂 Kelas: pushups
   ✅ OK: v_PushUps_g01_c01.avi (55 frames)
   ✅ OK: v_PushUps_g01_c02.avi (53 frames)
   ✅ OK: v_PushUps_g01_c03.avi (51 frames)
   ✅ OK: v_PushUps_g01_c04.avi (104 frames)
   ✅ OK: v_PushUps_g01_c05.avi (63 frames)
   ✅ OK: v_PushUps_g02_c01.avi (66 frames)
   ✅ OK: v_PushUps_g02_c02.avi (73 frames)
   ✅ OK: v_PushUps_g02_c03.avi (79 frames)
   ✅ OK: v_PushUps_g03_c01.avi (72 frames)
   ✅ OK: v_PushUps_g02_c04.avi (84 frames)
   ✅ OK: v_PushUps_g03_c02.avi (63 frames)
   ✅ OK: v_PushUps_g03_c03.avi (63 frames)
   ✅ OK: v_PushUps_g03_c04.avi (72 frames)
   ✅ OK: v_PushUps_g04_c01.avi (90 frames)
   ✅ OK: v_PushUps_g04_c02.avi (97 frames)
   ✅ OK: v_PushUps_g04_c03.avi (111 frames)
   ✅ OK: v_PushUps_g04_c05.avi (134 frames)
   ✅ OK: v_PushUps_g04_c04.avi (130 frames)
   ✅ OK: v_PushUps_g05_c01.avi (150 frames)
   ✅ OK: v_PushUps_g05_c02.avi (150 frames)
   ✅ OK: v_PushUps_g05_c03.avi (150 frames)
   ✅ OK: v_PushUps_g05_c04.avi (150 frames)
   ✅ OK: v_PushUps_g06_c01.avi (37 frames)
   ❌ Gagal/Kosong: v_PushUps_g06_c02.avi
   ✅ OK: v_PushUps_g06_c03.avi (1 frames)
   ✅ OK: v_PushUps_g06_c04.avi (6 frames)
   ✅ OK: v_PushUps_g07_c01.avi (74 frames)
   ✅ OK: v_PushUps_g07_c02.avi (77 frames)
   ✅ OK: v_PushUps_g07_c03.avi (78 frames)
   ✅ OK: v_PushUps_g07_c04.avi (77 frames)
   📂 Kelas: squat
   ✅ OK: v_BodyWeightSquats_g01_c01.avi (266 frames)
   ✅ OK: v_BodyWeightSquats_g01_c02.avi (252 frames)
   ✅ OK: v_BodyWeightSquats_g01_c03.avi (250 frames)
   ✅ OK: v_BodyWeightSquats_g01_c04.avi (174 frames)
   ✅ OK: v_BodyWeightSquats_g02_c02.avi (57 frames)
   ✅ OK: v_BodyWeightSquats_g02_c01.avi (102 frames)
   ✅ OK: v_BodyWeightSquats_g02_c03.avi (117 frames)
   ✅ OK: v_BodyWeightSquats_g02_c04.avi (121 frames)
   ✅ OK: v_BodyWeightSquats_g03_c01.avi (130 frames)
   ✅ OK: v_BodyWeightSquats_g03_c02.avi (52 frames)
   ✅ OK: v_BodyWeightSquats_g03_c03.avi (48 frames)
   ✅ OK: v_BodyWeightSquats_g03_c04.avi (101 frames)
   ✅ OK: v_BodyWeightSquats_g04_c01.avi (130 frames)
   ✅ OK: v_BodyWeightSquats_g03_c05.avi (75 frames)
   ✅ OK: v_BodyWeightSquats_g04_c02.avi (59 frames)
   ✅ OK: v_BodyWeightSquats_g04_c03.avi (87 frames)
   ✅ OK: v_BodyWeightSquats_g04_c04.avi (62 frames)
   ✅ OK: v_BodyWeightSquats_g05_c04.avi (260 frames)
   ✅ OK: v_BodyWeightSquats_g06_c02.avi (122 frames)
   ✅ OK: v_BodyWeightSquats_g06_c01.avi (179 frames)
   ✅ OK: v_BodyWeightSquats_g06_c03.avi (152 frames)
   ✅ OK: v_BodyWeightSquats_g06_c04.avi (95 frames)
   ✅ OK: v_BodyWeightSquats_g06_c05.avi (177 frames)
   ✅ OK: v_BodyWeightSquats_g07_c02.avi (103 frames)
   ✅ OK: v_BodyWeightSquats_g07_c01.avi (123 frames)
   ✅ OK: v_BodyWeightSquats_g07_c04.avi (54 frames)
   ✅ OK: v_BodyWeightSquats_g07_c03.avi (83 frames)
   ✅ OK: v_BodyWeightSquats_g05_c01.avi (71 frames)
   ✅ OK: v_BodyWeightSquats_g05_c03.avi (54 frames)
   ✅ OK: v_BodyWeightSquats_g05_c02.avi (135 frames)

🎥 MEMPROSES HMDB51 - SITUP...
   📄 Menggunakan Split File: situp_test_split1.txt
   ✅ OK: 5_Min_Tone_Abs_Workout_2__Fitness_Training_w__Tammy_situp_f_nm_np1_fr_goo_2.avi (97 frames)
   ✅ OK: 5_Min_Tone_Abs_Workout_2__Fitness_Training_w__Tammy_situp_f_nm_np1_fr_goo_3.avi (108 frames)
   ✅ OK: 5_Min_Tone_Abs_Workout_2__Fitness_Training_w__Tammy_situp_f_nm_np1_fr_goo_4.avi (101 frames)
   ✅ OK: 5_Min_Tone_Abs_Workout_2__Fitness_Training_w__Tammy_situp_f_nm_np1_le_goo_0.avi (105 frames)
   ✅ OK: 5_Min_Tone_Abs_Workout_2__Fitness_Training_w__Tammy_situp_f_nm_np1_le_goo_1.avi (101 frames)
   ✅ OK: 5_Min_Tone_Abs_Workout_2__Fitness_Training_w__Tammy_situp_f_nm_np1_le_goo_5.avi (110 frames)
   ✅ OK: 6_Minute_Abs_Routine_situp_f_nm_np1_ri_bad_0.avi (79 frames)
   ✅ OK: 6_Minute_Abs_Routine_situp_f_nm_np1_ri_bad_3.avi (79 frames)
   ✅ OK: 6_Minute_Abs_Routine_situp_f_nm_np1_ri_bad_4.avi (79 frames)
   ✅ OK: 6_Minute_Abs_Routine_situp_f_nm_np2_le_bad_1.avi (79 frames)
   ✅ OK: 6_Minute_Abs_Routine_situp_f_nm_np2_le_bad_2.avi (67 frames)
   ✅ OK: 6_Minute_Abs_Routine_situp_f_nm_np2_le_bad_5.avi (79 frames)
   ✅ OK: AKC_coach_Torrin_Rhodan_showing_situp_kettlebell_movements_situp_f_cm_np1_le_goo_0.avi (104 frames)
   ✅ OK: AKC_coach_Torrin_Rhodan_showing_situp_kettlebell_movements_situp_f_cm_np1_le_goo_1.avi (100 frames)
   ✅ OK: AKC_coach_Torrin_Rhodan_showing_situp_kettlebell_movements_situp_f_cm_np1_le_goo_2.avi (102 frames)
   ✅ OK: Ab_Workout__(_6_pack_abs_)_[_ab_exercises_for_ripped_abs_]_situp_f_nm_np1_le_goo_0.avi (79 frames)
   ✅ OK: Ab_Workout__(_6_pack_abs_)_[_ab_exercises_for_ripped_abs_]_situp_f_nm_np1_le_goo_1.avi (79 frames)
   ✅ OK: Ab_Workout__(_6_pack_abs_)_[_ab_exercises_for_ripped_abs_]_situp_f_nm_np1_le_goo_2.avi (79 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_cm_np1_ri_goo_0.avi (79 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_cm_np1_ri_goo_1.avi (79 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_cm_np1_ri_goo_2.avi (79 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_cm_np2_ri_goo_3.avi (57 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_nm_np3_ri_goo_4.avi (79 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_nm_np3_ri_goo_5.avi (138 frames)
   ✅ OK: Abs__Situps__Crunches_situp_u_nm_np3_ri_goo_6.avi (83 frames)
   ✅ OK: Army_situp_video_for_ROTC_cadets_situp_f_cm_np1_le_med_0.avi (78 frames)
   ✅ OK: Army_situp_video_for_ROTC_cadets_situp_f_cm_np1_le_med_1.avi (79 frames)
   ✅ OK: Army_situp_video_for_ROTC_cadets_situp_f_cm_np1_le_med_2.avi (79 frames)
   ✅ OK: Basic_Exercise_Plans_-_How_to_Perform_a_Sit-Up_situp_f_nm_np1_ri_goo_0.avi (189 frames)
   ✅ OK: Basic_Exercise_Plans_-_How_to_Perform_a_Sit-Up_situp_f_nm_np1_ri_goo_1.avi (130 frames)
   ✅ OK: Body_Flex_-_How2_-_Sit-Up_situp_f_nm_np1_le_goo_0.avi (93 frames)
   ✅ OK: Body_Flex_-_How2_-_Sit-Up_situp_f_nm_np1_le_goo_1.avi (99 frames)
   ✅ OK: Body_Flex_-_How2_-_Sit-Up_situp_f_nm_np1_le_goo_2.avi (109 frames)
   ✅ OK: Britney_Spears__Ab_Workout_Revealed_Video_situp_f_cm_np1_ri_goo_0.avi (126 frames)
   ✅ OK: Britney_Spears__Ab_Workout_Revealed_Video_situp_f_cm_np1_ri_goo_2.avi (141 frames)
   ✅ OK: Excercise_Demo_-_Proper_Sit-up_situp_f_nm_np1_ri_goo_0.avi (166 frames)
   ✅ OK: Excercise_Demo_-_Proper_Sit-up_situp_f_nm_np1_ri_goo_1.avi (162 frames)
   ✅ OK: Get_six_pack_abs_in_6_minutes_on_your_couch__This_Works!_situp_f_nm_np1_ba_med_0.avi (79 frames)
   ✅ OK: Get_six_pack_abs_in_6_minutes_on_your_couch__This_Works!_situp_f_nm_np1_ba_med_1.avi (79 frames)
   ✅ OK: Get_six_pack_abs_in_6_minutes_on_your_couch__This_Works!_situp_f_nm_np1_ba_med_2.avi (79 frames)
   ✅ OK: How_To_Workout_On_Vacation_-_How_to_Do_Crunches_Exercises_on_Vacation_situp_f_nm_np1_ba_goo_2.avi (93 frames)
   ✅ OK: How_To_Workout_On_Vacation_-_How_to_Do_Crunches_Exercises_on_Vacation_situp_f_nm_np1_le_goo_0.avi (131 frames)
   ✅ OK: How_To_Workout_On_Vacation_-_How_to_Do_Crunches_Exercises_on_Vacation_situp_f_nm_np1_le_goo_1.avi (93 frames)
   ✅ OK: How_to_Do_Proper_Situps_situp_f_cm_np1_le_goo_0.avi (76 frames)
   ✅ OK: How_to_Do_Proper_Situps_situp_f_cm_np1_le_goo_1.avi (83 frames)
   ✅ OK: How_to_Do_Proper_Situps_situp_f_cm_np1_le_goo_2.avi (78 frames)
   ✅ OK: How_to_Train_for_Boxing_-_Sit_Up_Techniques_for_Boxing_Training_situp_f_cm_np1_le_med_0.avi (78 frames)
   ✅ OK: How_to_Train_for_Boxing_-_Sit_Up_Techniques_for_Boxing_Training_situp_f_cm_np1_le_med_1.avi (78 frames)
   ✅ OK: How_to_Train_for_Boxing_-_Sit_Up_Techniques_for_Boxing_Training_situp_f_cm_np1_le_med_2.avi (78 frames)
   ✅ OK: How_to_get_a_six_pack_in_3minutes_EXPLOSIVE_RESULTS_IN_YOUR_4_WEEKS_situp_f_nm_np1_ri_med_0.avi (78 frames)
   ✅ OK: How_to_get_a_six_pack_in_3minutes_EXPLOSIVE_RESULTS_IN_YOUR_4_WEEKS_situp_f_nm_np1_ri_med_1.avi (79 frames)
   ✅ OK: How_to_get_a_six_pack_in_3minutes_EXPLOSIVE_RESULTS_IN_YOUR_4_WEEKS_situp_f_nm_np1_ri_med_2.avi (79 frames)
   ✅ OK: Incline_Dumbell_Situps_45lbs_2ndxpr_25x_situp_f_nm_np1_ri_med_0.avi (129 frames)
   ✅ OK: Incline_Dumbell_Situps_45lbs_2ndxpr_25x_situp_f_nm_np1_ri_med_1.avi (121 frames)
   ✅ OK: Intermidate_to_advanced_abs_situp_f_cm_np1_fr_med_1.avi (56 frames)
   ✅ OK: Intermidate_to_advanced_abs_situp_f_nm_np1_fr_med_0.avi (64 frames)
   ✅ OK: Intermidate_to_advanced_abs_situp_f_nm_np1_fr_med_2.avi (54 frames)
   ✅ OK: Mark_Pfeltz_Sets_World_Record_For_sit_ups_situp_f_nm_np1_fr_bad_2.avi (112 frames)
   ✅ OK: Mark_Pfeltz_Sets_World_Record_For_sit_ups_situp_f_nm_np1_fr_bad_3.avi (75 frames)
   ✅ OK: Mark_Pfeltz_Sets_World_Record_For_sit_ups_situp_f_nm_np1_fr_bad_4.avi (104 frames)
   ✅ OK: Mark_Pfeltz_Sets_World_Record_For_sit_ups_situp_f_nm_np1_ri_bad_0.avi (8 frames)
   ❌ Gagal/Kosong: Mark_Pfeltz_Sets_World_Record_For_sit_ups_situp_f_nm_np1_ri_bad_1.avi
   ✅ OK: Ninja_Warrior_Ultimate_Six_Pack_Abs_Sit_up_Training_situp_f_nm_np1_ri_med_0.avi (79 frames)
   ✅ OK: Ninja_Warrior_Ultimate_Six_Pack_Abs_Sit_up_Training_situp_f_nm_np1_ri_med_1.avi (79 frames)
   ✅ OK: Ninja_Warrior_Ultimate_Six_Pack_Abs_Sit_up_Training_situp_f_nm_np1_ri_med_2.avi (79 frames)
   ✅ OK: Octogen_-_CrossFit_Sydney_Glute_Ham_Situps_1_situp_f_nm_np1_ri_goo_0.avi (120 frames)
   ✅ OK: Octogen_-_CrossFit_Sydney_Glute_Ham_Situps_1_situp_f_nm_np1_ri_goo_1.avi (97 frames)
   ✅ OK: Octogen_-_CrossFit_Sydney_Glute_Ham_Situps_1_situp_f_nm_np1_ri_goo_2.avi (101 frames)
   ✅ OK: Personal_Training_Workout_Tips_situp_f_nm_np1_le_goo_0.avi (93 frames)
   ✅ OK: Personal_Training_Workout_Tips_situp_f_nm_np1_le_goo_1.avi (96 frames)
   ✅ OK: Rutina_de_Abdominales___Abs_Exercise_situp_f_cm_np1_ri_goo_0.avi (79 frames)
   ✅ OK: Rutina_de_Abdominales___Abs_Exercise_situp_f_cm_np1_ri_goo_1.avi (78 frames)
   ✅ OK: Rutina_de_Abdominales___Abs_Exercise_situp_f_cm_np1_ri_goo_2.avi (79 frames)
   ✅ OK: Sit_Ups_for_6pack_Abs___Forgotten_Abdominal_Exercise_situp_f_nm_np1_le_med_0.avi (91 frames)
   ✅ OK: Sit_Ups_for_6pack_Abs___Forgotten_Abdominal_Exercise_situp_f_nm_np1_le_med_1.avi (109 frames)
   ✅ OK: Sit_Ups_for_6pack_Abs___Forgotten_Abdominal_Exercise_situp_f_nm_np1_le_med_2.avi (104 frames)
   ✅ OK: Sit_ups_and_crunch_situp_f_nm_np1_le_goo_0.avi (94 frames)
   ✅ OK: Sit_ups_and_crunch_situp_f_nm_np1_le_goo_1.avi (89 frames)
   ✅ OK: Sit_ups_and_crunch_situp_f_nm_np1_le_goo_2.avi (94 frames)
   ✅ OK: Sit_ups_situp_f_nm_np1_ri_goo_0.avi (78 frames)
   ✅ OK: Sit_ups_situp_f_nm_np1_ri_goo_1.avi (84 frames)
   ✅ OK: Sit_ups_situp_f_nm_np1_ri_goo_2.avi (83 frames)
   ✅ OK: Tiger_Abs_-_Triple_Sit_Ups_situp_f_nm_np1_le_goo_0.avi (86 frames)
   ✅ OK: Tiger_Abs_-_Triple_Sit_Ups_situp_f_nm_np1_le_goo_1.avi (91 frames)
   ✅ OK: Tiger_Abs_-_Triple_Sit_Ups_situp_f_nm_np1_le_goo_2.avi (82 frames)
   ✅ OK: Timed_situps_1_minute_situp_f_nm_np1_ri_med_0.avi (79 frames)
   ✅ OK: Timed_situps_1_minute_situp_f_nm_np1_ri_med_1.avi (79 frames)
   ✅ OK: Timed_situps_1_minute_situp_f_nm_np1_ri_med_2.avi (79 frames)
   ✅ OK: Waschbrettbauch_I-__bung_von_fitness_com_situp_f_nm_np1_ba_goo_0.avi (82 frames)
   ✅ OK: Waschbrettbauch_I-__bung_von_fitness_com_situp_f_nm_np1_ba_goo_1.avi (108 frames)
   ✅ OK: crazy_105_sit-ups_situp_f_cm_np2_le_med_0.avi (55 frames)
   ✅ OK: crazy_105_sit-ups_situp_f_cm_np2_le_med_1.avi (62 frames)
   ✅ OK: kettlebell_training_for_pack_ABS_-_workout_situp_f_nm_np1_le_goo_1.avi (117 frames)
   ✅ OK: kettlebell_training_for_pack_ABS_-_workout_situp_f_nm_np1_le_goo_2.avi (119 frames)
   ✅ OK: situps_situp_f_cm_np1_le_med_1.avi (78 frames)
   ✅ OK: situps_situp_f_cm_np1_le_med_2.avi (79 frames)
   ✅ OK: situps_situp_f_cm_np3_ri_med_4.avi (78 frames)
   ✅ OK: situps_situp_f_cm_np3_ri_med_5.avi (79 frames)
   ✅ OK: situps_situp_f_cm_np3_ri_med_6.avi (79 frames)
   ✅ OK: situps_situp_f_nm_np1_le_med_0.avi (78 frames)

🎉 SELESAI! Dataset NPY Final (45 Fitur) siap di:
/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Dataset_NPY_Final_45Fitur


Kode Training & Studi Ablasi

In [ ]:
# KODE TRAINING DAN STUDI ABLASI CNN-LSTM

import os, random
import numpy as np
import tensorflow as tf
from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, LSTM,
                                     Dense, Dropout, BatchNormalization, Activation, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import pandas as pd


#SEED untuk memastikan semua proses menghasilkan urutan yang sama setiap kali kode dijalankan
#Penting agar hasil eksperimen dapat direproduksi dan perbandingan antar skenario ablasi menjadi adil
SEED_VALUE = 42
os.environ['PYTHONHASHSEED'] = str(SEED_VALUE)
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)
print("Seed 42 terkunci")

# KONFIGURASI FOLDER
PATH_DATASET_ROOT = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Dataset_NPY_Final_45Fitur'
PATH_SAVE_ROOT    = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Hasil_Ablation_Murni_Final'
os.makedirs(PATH_SAVE_ROOT, exist_ok=True)

CLASSES_LIST = ["jumpingjack", "lunges", "pushups", "squat", "Situp", "idle"]
NUM_CLASSES  = len(CLASSES_LIST)
SEQ_LEN      = 30
EPOCHS       = 100
BATCH_SIZE   = 32


#AUGMENTASI DATA (Sebelum Fitur Engineering)
def aug_flip(raw):
    f = raw.copy()
    for j in range(12):
        f[:, j*3] = 1.0 - f[:, j*3]
    COORD_FLIP_PAIRS = [(0,1),(2,3),(4,5),(6,7),(8,9),(10,11)]
    for li, ri in COORD_FLIP_PAIRS:
        lc = slice(li*3, li*3+3)
        rc = slice(ri*3, ri*3+3)
        f[:, lc], f[:, rc] = f[:, rc].copy(), f[:, lc].copy()
    f[:, 38], f[:, 39] = raw[:, 39].copy(), raw[:, 38].copy()
    f[:, 40], f[:, 41] = raw[:, 41].copy(), raw[:, 40].copy()
    return f

def time_warp(sequence, sigma=0.2):
    orig_steps   = np.arange(sequence.shape[0])
    knot_steps   = np.linspace(0, sequence.shape[0] - 1, num=6)
    random_warps = np.random.normal(loc=1.0, scale=sigma, size=(6,))
    cs           = CubicSpline(knot_steps, random_warps)
    warp_curve   = cs(orig_steps)
    new_steps    = np.cumsum(warp_curve)
    new_steps    = new_steps - new_steps[0]
    new_steps    = new_steps / new_steps[-1] * (sequence.shape[0] - 1)
    ret = np.zeros_like(sequence)
    for dim in range(sequence.shape[1]):
        ret[:, dim] = np.interp(orig_steps, new_steps, sequence[:, dim])
    return ret

def augment_45(window_45, class_name):
    if class_name == 'idle':
        return window_45 + np.random.normal(0, 0.002, window_45.shape)
    aug = window_45.copy()
    if np.random.rand() < 0.5: aug = aug_flip(aug)
    if np.random.rand() < 0.5: aug = time_warp(aug)
    if np.random.rand() < 0.5: aug += np.random.normal(0, 0.005, aug.shape)
    return aug

#FEATURE ENGINEERING (KINEMATIKA)
def compute_kinematics(window, smooth_sigma=0.3):
    smoothed = np.zeros_like(window)
    for j in range(window.shape[1]):
        smoothed[:, j] = gaussian_filter1d(window[:, j], sigma=smooth_sigma)
    vel = np.diff(smoothed, axis=0, prepend=smoothed[0:1])
    acc = np.diff(vel,      axis=0, prepend=vel[0:1])
    return np.concatenate([window, vel, acc], axis=1)


#LOAD DATA
def load_data(mode, use_raw36=False, use_fe=True, use_aug=False):
    features, labels = [], []
    class_map  = {c: i for i, c in enumerate(CLASSES_LIST)}
    split_path = os.path.join(PATH_DATASET_ROOT, mode)
    stride     = SEQ_LEN // 2 if mode == 'train' else SEQ_LEN #stride: 15 train (overlap 50%), 30 test()

    print(f"\n[LOAD] mode={mode} | raw36={use_raw36} | FE={use_fe} | Aug={use_aug} | stride={stride}")

    for class_name in CLASSES_LIST:
        class_dir = os.path.join(split_path, class_name)
        if not os.path.exists(class_dir):
            continue

        # FIX SORTED: Agar urutan data dibaca sama terus
        files = [os.path.join(class_dir, f) for f in sorted(os.listdir(class_dir)) if f.endswith('.npy')]
        win_count = 0

        for fpath in files:
            data = np.load(fpath)
            if data.shape[0] < SEQ_LEN:
                continue

            for i in range(0, data.shape[0] - SEQ_LEN + 1, stride):
                if use_raw36:
                    window = data[i: i+SEQ_LEN, :36].copy()
                else:
                    window = data[i: i+SEQ_LEN, :45].copy()

                if use_aug and mode == 'train':
                    aug_win = augment_45(
                        window if not use_raw36 else np.concatenate([window, np.zeros((SEQ_LEN, 9))], axis=1),
                        class_name)
                    aug_win = aug_win if not use_raw36 else aug_win[:, :36]
                    aug_feat = compute_kinematics(aug_win) if use_fe else aug_win
                    features.append(aug_feat)
                    labels.append(class_map[class_name])

                feat = compute_kinematics(window) if use_fe else window
                features.append(feat)
                labels.append(class_map[class_name])
                win_count += 1

        print(f"  {class_name}: {len(files)} file → {win_count} windows")

    X = np.array(features, dtype=np.float32)
    y = np.array(labels,   dtype=np.int32)
    print(f"  TOTAL: X={X.shape} | y={y.shape}")
    return X, y


# NORMALISASI DATA (ROBUST SCALING)
def normalize(X_train, X_val, X_test):
    median = np.median(X_train, axis=(0, 1), keepdims=True)
    q75    = np.percentile(X_train, 75, axis=(0, 1))
    q25    = np.percentile(X_train, 25, axis=(0, 1))
    iqr    = (q75 - q25) + 1e-8
    X_tr   = (X_train - median) / iqr
    X_v    = (X_val   - median) / iqr
    X_te   = (X_test  - median) / iqr
    return X_tr, X_v, X_te, median, iqr


# ARSITEKTUR
def build_cnn_only(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x   = Conv1D(128, 5, padding='same', kernel_regularizer=l2(1e-4))(inp)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = MaxPooling1D(2)(x);      x = Dropout(0.3)(x)
    x   = Conv1D(256, 3, padding='same', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = GlobalAveragePooling1D()(x); x = Dropout(0.3)(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x);     x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='CNN_Only')

def build_lstm_only(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x   = LSTM(256, return_sequences=True, kernel_regularizer=l2(1e-4))(inp)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = LSTM(128, return_sequences=False, kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='LSTM_Only')

def build_cnn_lstm(input_shape, num_classes):
    inp = Input(shape=input_shape)
    x   = Conv1D(128, 5, padding='same', kernel_regularizer=l2(1e-4))(inp)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = MaxPooling1D(2)(x);      x = Dropout(0.3)(x)
    x   = Conv1D(256, 3, padding='same', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Activation('relu')(x)
    x   = MaxPooling1D(2)(x);      x = Dropout(0.3)(x)
    x   = LSTM(256, return_sequences=True, kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = LSTM(128, return_sequences=False, kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x); x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out, name='CNN_LSTM')

#TEMPORAL ENSEMBLE
def temporal_ensemble(y_pred_probs, window_size=5):
    smoothed = np.copy(y_pred_probs)
    for i in range(len(y_pred_probs)):
        start = max(0, i - window_size + 1)
        smoothed[i] = np.mean(y_pred_probs[start:i+1], axis=0)
    return np.argmax(smoothed, axis=1)

#PLOT/VISUALISASI
def plot_results(history, y_test, y_pred_raw, y_pred_ens, case_id, case_name, acc_raw, acc_ens, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history.history['accuracy'],     label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'K{case_id} Accuracy\nRaw={acc_raw:.2f}% | Ens={acc_ens:.2f}%')
    axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history.history['loss'],     label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'K{case_id} Loss')
    axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

    cm = confusion_matrix(y_test, y_pred_ens)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST)
    axes[2].set_title(f'K{case_id} CM (Ensemble: {acc_ens:.2f}%)')
    axes[2].set_ylabel('Aktual'); axes[2].set_xlabel('Prediksi')

    plt.tight_layout()
    path = os.path.join(save_dir, f'kasus{case_id}_murni_plot.png')
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  [SAVED] Plot → {path}")


#TRAIN PER KASUS
def train_case(case_id, case_name, build_fn, use_raw36, use_fe, use_aug):
    print(f"\n{'='*65}")
    print(f"  MEMULAI KASUS {case_id}: {case_name}")
    print(f"{'='*65}")

    save_dir = os.path.join(PATH_SAVE_ROOT, f'kasus{case_id}')
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f'kasus{case_id}_murni_best.keras')
  #LOAD DATA
    X_tr_raw, y_tr = load_data('train', use_raw36, use_fe, use_aug)
    X_te_raw, y_te = load_data('test',  use_raw36, use_fe, False)

    if len(X_tr_raw) == 0:
        print(f"  [ERROR] Dataset kosong!")
        return None
  #SPLIT TRAIN-VALIDATION (85:15)
    X_tr, X_val, y_tr_s, y_val_s = train_test_split(
        X_tr_raw, y_tr, test_size=0.15, random_state=SEED_VALUE, stratify=y_tr)
  #NORMALISASI DATA
    X_tr, X_val, X_te, median, iqr = normalize(X_tr, X_val, X_te_raw)
    print(f"  [INFO] Train={len(X_tr)} | Val={len(X_val)} | Test={len(X_te)}")

    if case_id in [6, 7]:
        np.save(os.path.join(save_dir, 'scaler_median.npy'), median)
        np.save(os.path.join(save_dir, 'scaler_iqr.npy'),    iqr)
        print(f"  [SAVED] Scaler (Penting buat Real-Time) tersimpan di → {save_dir}")
  #CLASS WEIGHTING MENGATASI KETIDAKSEIMBANGAN DATA
    weights = compute_class_weight('balanced', classes=np.unique(y_tr_s), y=y_tr_s)
    cw_dict = dict(enumerate(weights))
  #One-hot encoding
    y_tr_cat  = to_categorical(y_tr_s,  NUM_CLASSES)
    y_val_cat = to_categorical(y_val_s, NUM_CLASSES)
    y_te_cat  = to_categorical(y_te,    NUM_CLASSES)

    # FIX CLEAR SESSION: Bersihkan RAM GPU dari model sebelumnya
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED_VALUE)
    np.random.seed(SEED_VALUE)

    #build model
    input_shape = (SEQ_LEN, X_tr.shape[2])
    model = build_fn(input_shape, NUM_CLASSES)

    model.compile(optimizer=AdamW(learning_rate=0.001, weight_decay=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    cb_list = [
        ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
    ]
    #TRAINING
    history = model.fit(
        X_tr, y_tr_cat, validation_data=(X_val, y_val_cat),
        epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=cw_dict,
        callbacks=cb_list, verbose=1)
    #EVALUASI
    best_model   = tf.keras.models.load_model(model_path)
    y_pred_probs = best_model.predict(X_te, verbose=0)
    y_pred_raw   = np.argmax(y_pred_probs, axis=1)
    y_pred_ens   = temporal_ensemble(y_pred_probs, window_size=5)

    acc_raw = np.mean(y_pred_raw == y_te) * 100
    acc_ens = np.mean(y_pred_ens == y_te) * 100

    print(f"\n  [RESULT] Kasus {case_id} — Raw: {acc_raw:.2f}% | Ensemble: {acc_ens:.2f}%")
    print(classification_report(y_te, y_pred_ens, target_names=CLASSES_LIST))

    pd.DataFrame(history.history).to_csv(os.path.join(save_dir, f'kasus{case_id}_murni_history.csv'), index=False)
    plot_results(history, y_te, y_pred_raw, y_pred_ens, case_id, case_name, acc_raw, acc_ens, save_dir)

    return {'kasus': case_id, 'nama': case_name, 'raw36': use_raw36, 'FE': use_fe, 'Aug': use_aug,
            'n_feat': X_tr.shape[2], 'acc_raw': f'{acc_raw:.2f}%', 'acc_ensemble': f'{acc_ens:.2f}%'}

#MAIN
if __name__ == '__main__':

    # Pilih kasus yang akan dijalankan (1-7)
    # Untuk training model final, gunakan 7
    TARGET_KASUS = 7

    ABLATION_CASES = [
        (1, 'CNN-Only  | 36 raw + FE = 108 fitur | No Aug', build_cnn_only,  True,  True, False),
        (2, 'LSTM-Only | 36 raw + FE = 108 fitur | No Aug', build_lstm_only, True,  True, False),
        (3, 'CNN-LSTM  | 36 raw + FE = 108 fitur | No Aug', build_cnn_lstm,  True,  True, False),
        (4, 'CNN-Only  | 45 FE = 135 fitur | No Aug',       build_cnn_only,  False, True,  False),
        (5, 'LSTM-Only | 45 FE = 135 fitur | No Aug',       build_lstm_only, False, True,  False),
        (6, 'CNN-LSTM  | 45 FE = 135 fitur | No Aug',       build_cnn_lstm,  False, True,  False),
        (7, 'CNN-LSTM  | 45 FE = 135 fitur | Aug',          build_cnn_lstm,  False, True,  True),
    ]

    print(f"\n{'='*65}")
    print(f"MENJALANKAN MODE MURNI UNTUK KASUS {TARGET_KASUS}")
    print(f"{'='*65}")

    for case_data in ABLATION_CASES:
        if case_data[0] == TARGET_KASUS:
            cid, cname, bfn, ur36, ufe, uaug = case_data
            result = train_case(cid, cname, bfn, ur36, ufe, uaug)
            print(f"\n EKSEKUSI KASUS {TARGET_KASUS} SELESAI!")
            break

In [ ]:
K7
=================================================================
  MEMULAI KASUS 7: CNN-LSTM  | 45 FE = 135 fitur | Aug
=================================================================

[LOAD] mode=train | raw36=False | FE=True | Aug=True | stride=15
  jumpingjack: 86 file → 361 windows
  lunges: 90 file → 1153 windows
  pushups: 72 file → 255 windows
  squat: 82 file → 594 windows
  Situp: 69 file → 319 windows
  idle: 84 file → 288 windows
  TOTAL: X=(5940, 30, 135) | y=(5940,)

[LOAD] mode=test | raw36=False | FE=True | Aug=False | stride=30
  jumpingjack: 37 file → 78 windows
  lunges: 37 file → 235 windows
  pushups: 29 file → 70 windows
  squat: 30 file → 107 windows
  Situp: 30 file → 78 windows
  idle: 29 file → 47 windows
  TOTAL: X=(615, 30, 135) | y=(615,)
  [INFO] Train=5049 | Val=891 | Test=615

  [RESULT] Kasus 7 — Raw: 89.27% | Ensemble: 94.96%
              precision    recall  f1-score   support

 jumpingjack       0.97      0.90      0.93        78
      lunges       0.97      0.99      0.98       235
     pushups       0.96      0.97      0.96        70
       squat       0.97      0.93      0.95       107
       Situp       0.97      0.86      0.91        78
        idle       0.77      0.98      0.86        47

    accuracy                           0.95       615
   macro avg       0.93      0.94      0.93       615
weighted avg       0.95      0.95      0.95       615


KODE IMPLEMENTASI REAL-TIME

In [ ]:
import os
import sys
import cv2
import time
import mediapipe as mp
import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, Counter
from scipy.ndimage import gaussian_filter1d
from tensorflow.keras.models import load_model


#KONFIGURASI MODEL DAN ROBUST SCALER
MODEL_PATH = 'assets/kasus7_murni_best.keras'
SCALER_MEDIAN_PATH = 'assets/scaler_median_7.npy'
SCALER_IQR_PATH = 'assets/scaler_iqr_7.npy'

CLASSES_LIST = ["jumpingjack", "lunges", "pushups", "squat", "Situp", "idle"]
SEQUENCE_LENGTH = 30
CONFIDENCE_THRESHOLD = 0.65
MOTION_THRESHOLD = 0.010

COLORS = {
    'jumpingjack': (0, 165, 255),
    'lunges': (0, 255, 255),
    'pushups': (0, 255, 0),
    'squat': (255, 0, 255),
    'Situp': (0, 100, 255),
    'idle': (200, 200, 200),
    'OUT_OF_FRAME': (0, 0, 255),
    'MENGANALISIS...': (255, 255, 0),
    'BUFFERING': (255, 150, 0)
}

LOG_DIR = 'output/logs'
PLOT_DIR = 'output/plots'



#LOGIKA DASAR & EKSTRAKSI FITUR (BIOMEKANIK)
def check_visibility(landmarks):
    shoulder_vis = (landmarks[11].visibility + landmarks[12].visibility) / 2
    hip_vis = (landmarks[23].visibility + landmarks[24].visibility) / 2
    knee_vis = (landmarks[25].visibility + landmarks[26].visibility) / 2
    return shoulder_vis > 0.5 and hip_vis > 0.5 and knee_vis > 0.5


def calculate_motion_score(window_data):
    # Pantau pergerakan Bahu (1), Pinggul (19), Lutut (25), dan Engkel (31)
    # Menentukan apakah model diinferensi atatu tidak
    return np.mean([
        np.std(window_data[:, 1]),  # BAHU
        np.std(window_data[:, 19]),  # Pinggul
        np.std(window_data[:, 25]),  # Lutut
        np.std(window_data[:, 31])  # Engkel
    ])


def calculate_angle(a, b, c):
    #Menghitung sudut antara tiga titik (a, b, c) dengan b sebagai titik pusat.
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))


def extract_features_45(landmarks):
    """
    Mengekstraksi 45 fitur dari 12 landmark utama MediaPipe.
    Input: 33 landmark MediaPipe
    Output: 45 fitur (36 koordinat mentah + 9 fitur geometris)
    """
    indices = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
    frame_raw, xs, ys = [], [], []
    for idx in indices:
        lm = landmarks[idx]
        frame_raw.extend([lm.x, lm.y, lm.z])
        xs.append(lm.x);
        ys.append(lm.y)
    frame_raw = np.array(frame_raw)

    def get_pt(idx): return np.array([frame_raw[idx], frame_raw[idx + 1]])

    l_sh, r_sh = get_pt(0), get_pt(3)
    l_hip, r_hip = get_pt(18), get_pt(21)
    l_knee, r_knee = get_pt(24), get_pt(27)
    l_ank, r_ank = get_pt(30), get_pt(33)

    ankle_dist = np.abs(frame_raw[30] - frame_raw[33])
    knee_ver_diff = np.abs(frame_raw[25] - frame_raw[28])

    ang_l_knee = calculate_angle(l_hip, l_knee, l_ank)
    ang_r_knee = calculate_angle(r_hip, r_knee, r_ank)
    ang_l_hip = calculate_angle(l_sh, l_hip, l_knee)
    ang_r_hip = calculate_angle(r_sh, r_hip, r_knee)

    mid_shoulder = (l_sh + r_sh) / 2
    mid_hip = (l_hip + r_hip) / 2
    trunk_vec = mid_shoulder - mid_hip
    norm_trunk = np.linalg.norm(trunk_vec) + 1e-8
    dot_product = np.dot(trunk_vec, np.array([0, -1]))
    feat_trunk = np.degrees(np.arccos(np.clip(dot_product / norm_trunk, -1.0, 1.0))) / 180.0

    feat_knee_sym = np.abs(ang_l_knee - ang_r_knee) / 180.0
    width, height = max(xs) - min(xs), max(ys) - min(ys)
    feat_ratio = min(height / (width + 0.001), 3.0) / 3.0
    norm_angles = [ang_l_knee / 180.0, ang_r_knee / 180.0, ang_l_hip / 180.0, ang_r_hip / 180.0]

    return np.concatenate(
        [frame_raw, [ankle_dist, knee_ver_diff], norm_angles, [feat_trunk, feat_knee_sym, feat_ratio]])



#INFERENSI MODEL(CNN-LSTM)
class ExerciseModel:
    def __init__(self):
        self.model = load_model(MODEL_PATH, compile=False)
        self.scaler_median = np.load(SCALER_MEDIAN_PATH).squeeze()
        self.scaler_iqr = np.load(SCALER_IQR_PATH).squeeze()

    def compute_135_features(self, window, smooth_sigma=0.3):
        if smooth_sigma > 0:
            smoothed = np.zeros_like(window)
            for j in range(window.shape[1]):
                smoothed[:, j] = gaussian_filter1d(window[:, j], sigma=smooth_sigma)
        else:
            smoothed = window
        velocity = np.diff(smoothed, axis=0, prepend=smoothed[0:1])
        acceleration = np.diff(velocity, axis=0, prepend=velocity[0:1])
        return np.concatenate([window, velocity, acceleration], axis=-1)

    def predict(self, sequence_data):
        window_135 = self.compute_135_features(sequence_data)
        window_norm = (window_135 - self.scaler_median) / (self.scaler_iqr + 1e-6)
        input_data = np.expand_dims(window_norm, axis=0)
        prediction = self.model.predict(input_data, verbose=0)
        max_idx = np.argmax(prediction[0])
        pred_conf = np.max(prediction[0])
        return max_idx, float(pred_conf)


#VISUALISASI DAN UI PROGRAM
CUSTOM_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24), (23, 25), (25, 27),
    (24, 26), (26, 28)
]
USED_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]


def draw_custom_skeleton(frame, landmarks):
    #Menggambar kerangka tubuh pada frame berdasarkan landmark yang terdeteksi.
    h, w, _ = frame.shape
    for connection in CUSTOM_CONNECTIONS:
        idx1, idx2 = connection
        lm1, lm2 = landmarks[idx1], landmarks[idx2]
        if lm1.visibility > 0.5 and lm2.visibility > 0.5:
            pt1 = (int(lm1.x * w), int(lm1.y * h))
            pt2 = (int(lm2.x * w), int(lm2.y * h))
            cv2.line(frame, pt1, pt2, (255, 255, 255), 3)

    for idx in USED_INDICES:
        lm = landmarks[idx]
        if lm.visibility > 0.5:
            pt = (int(lm.x * w), int(lm.y * h))
            cv2.circle(frame, pt, 6, (0, 0, 255), -1)


def draw_ui_klasifikasi(frame, status, fps, inf_time, mp_time, queue_size, waktu_berjalan):
    h, w, _ = frame.shape
    overlay = frame.copy()

    cv2.rectangle(overlay, (0, 0), (w, 95), (15, 15, 15), -1)
    cv2.rectangle(overlay, (0, h - 40), (w, h), (15, 15, 15), -1)
    frame = cv2.addWeighted(overlay, 0.8, frame, 0.2, 0)

    #Label Prediksi
    color = COLORS.get(status['class'], (255, 255, 255))
    main_text = "SIAP (IDLE)" if status['class'] == 'idle' else status['class'].upper()
    if status['class'] in ['OUT_OF_FRAME', 'MENGANALISIS...', 'BUFFERING']:
        main_text = status['message'] or status['class']

    cv2.putText(frame, "PREDIKSI AI:", (20, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    font_scale = 0.9 if len(main_text) > 15 else 1.1
    cv2.putText(frame, main_text, (20, 60), cv2.FONT_HERSHEY_DUPLEX, font_scale, color, 2)
    #Menampilkan Confidence Score
    if status['class'] not in ['OUT_OF_FRAME', 'MENGANALISIS...', 'BUFFERING', 'idle']:
        cv2.putText(frame, f"Conf: {status['confidence'] * 100:.1f}%", (320, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color,
                    2)
    #TIMER
    menit, detik = int(waktu_berjalan // 60), int(waktu_berjalan % 60)
    cv2.putText(frame, f"TIMER: {menit:02d}:{detik:02d}", (w // 2 - 100, 55), cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 0, 255),
                2)
    #Metrik performa (FPS, MediaPipe time,AI infer time)
    cv2.putText(frame, f"FPS       : {int(fps)}", (w - 280, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    cv2.putText(frame, f"MP Time   : {int(mp_time)} ms", (w - 280, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    cv2.putText(frame, f"AI Infer  : {int(inf_time)} ms", (w - 280, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255),
                1)

    cv2.circle(frame, (w - 80, 25), 8, (0, 0, 255), -1)
    cv2.putText(frame, "REC", (w - 65, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    if queue_size < SEQUENCE_LENGTH and status['class'] not in ['OUT_OF_FRAME']:
        cv2.rectangle(frame, (0, 95), (int(w * (queue_size / SEQUENCE_LENGTH)), 100), (255, 150, 0), -1)

    cv2.putText(frame, "Sistem HAR Full AI | 'S' Toggle Skeleton | 'Q' Keluar", (20, h - 15), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (200, 200, 200), 1)
    return frame


def generate_thesis_plots(csv_path):
    target_dir = os.path.dirname(csv_path)
    if not target_dir: target_dir = "."
    base_name = os.path.basename(csv_path).replace('.csv', '')

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f" Gagal membaca CSV untuk grafik: {e}")
        return

    sns.set_theme(style="whitegrid")

    # 1. TIMELINE PREDIKSI
    plt.figure(figsize=(12, 5))
    mapping = {'IDLE': 0, 'JUMPINGJACK': 1, 'SQUAT': 2, 'PUSHUPS': 3, 'LUNGES': 4, 'SITUP': 5, 'BUFFERING': 6,
               'OUT_OF_FRAME': 6}
    df['ID_Tebakan'] = df['Tebakan'].str.upper().map(mapping).fillna(6)
    plt.step(df['Waktu (Detik)'], df['ID_Tebakan'], where='post', color='#2c3e50', linewidth=2)
    plt.yticks(range(7), ['IDLE', 'JJ', 'SQUAT', 'PUSHUP', 'LUNGES', 'SITUP', 'OTHER'])
    plt.title(f'Timeline Prediksi Gerakan AI - {base_name}', fontsize=14)
    plt.xlabel('Waktu (Detik)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_1_timeline.png"), dpi=300)
    plt.close()

    # 2. CONFIDENCE TREND
    plt.figure(figsize=(12, 4))
    plt.plot(df['Waktu (Detik)'], df['Confidence'], color='#e67e22', linewidth=1.5)
    plt.fill_between(df['Waktu (Detik)'], df['Confidence'], color='#e67e22', alpha=0.2)
    plt.ylim(0, 1.1)
    plt.title(f'Fluktuasi Confidence Level - {base_name}', fontsize=14)
    plt.ylabel('Confidence (0-1)')
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_2_confidence.png"), dpi=300)
    plt.close()

    # 3. FPS & RAM
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.set_xlabel('Waktu (Detik)')
    ax1.set_ylabel('FPS', color='#27ae60')
    ax1.plot(df['Waktu (Detik)'], df['FPS'], color='#27ae60', alpha=0.8)
    ax1.tick_params(axis='y', labelcolor='#27ae60')
    ax2 = ax1.twinx()
    ax2.set_ylabel('RAM Usage (MB)', color='#7f8c8d')
    ax2.fill_between(df['Waktu (Detik)'], df['RAM Usage (MB)'], color='#7f8c8d', alpha=0.2)
    ax2.tick_params(axis='y', labelcolor='#7f8c8d')
    plt.title(f'Analisis Performa Sistem - {base_name}', fontsize=14)
    fig.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_3_performance.png"), dpi=300)
    plt.close()

    # 4. LATENCY
    plt.figure(figsize=(8, 6))
    avg_mp = df['MediaPipe Time (ms)'].mean()
    avg_ai = df['AI Infer Time (ms)'].mean()
    plt.bar(['Latency'], [avg_mp], color='#3498db', label='MediaPipe')
    plt.bar(['Latency'], [avg_ai], bottom=[avg_mp], color='#e74c3c', label='AI LSTM')
    plt.ylabel('Waktu (ms)')
    plt.title(f'Distribusi Latency Komputasi - {base_name}', fontsize=14)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_4_latency.png"), dpi=300)
    plt.close()

    print(f" 4 Grafik metrik otomatis berhasil disimpan di: {target_dir}")



#FUNGSI UTAMA (MAIN LOOP CAMERA)
def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def main():
    clear_screen()
    print("=" * 65)
    print(" SISTEM HAR SKELETON-BASED (MODE FULL AI)".center(65))
    print("=" * 65)
    nama_subjek = input("Masukkan Nama Subjek (contoh: Andi_Testing): ")
    if not nama_subjek: nama_subjek = "Subjek_Test"

    # Setup Folder Penyimpanan
    base_dir = "Data_Pengujian"
    subject_dir = os.path.join(base_dir, nama_subjek)
    os.makedirs(subject_dir, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    file_prefix = f"FullAI_{timestamp}"
    csv_filename = os.path.join(subject_dir, f"{file_prefix}.csv")
    video_filename = os.path.join(subject_dir, f"{file_prefix}.mp4")

    print(f"\n Penyimpanan: {subject_dir}")
    print("Memuat AI Model CNN-LSTM...")

    # Inisialisasi Model AI
    ai_model = ExerciseModel()

    # Inisialisasi MediaPipe
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    # Inisialisasi Kamera
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    frame_width, frame_height = 1280, 720
    cap.set(cv2.CAP_PROP_FPS, 30)
    fps_kamera = 30.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(video_filename, fourcc, fps_kamera, (frame_width, frame_height))

    frames_queue = deque(maxlen=SEQUENCE_LENGTH)
    PREDICTION_HISTORY = deque(maxlen=9)
    log_performa = []

    current_status = {'class': 'BUFFERING', 'confidence': 0.0, 'message': 'MENGUMPULKAN DATA...'}
    prev_frame_time, frame_count = 0, 0
    show_skeleton = True
    process_memory = psutil.Process(os.getpid())

    window_name = f"Testing AI: {nama_subjek}"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1024, 768)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1
        new_frame_time = time.time()
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Proses MediaPipe
        t_mp_start = time.time()
        results = pose.process(rgb_frame)
        mp_time_ms = (time.time() - t_mp_start) * 1000

        inference_time_ms = 0
        ram_usage_mb = process_memory.memory_info().rss / (1024 * 1024)
        cpu_usage_percent = process_memory.cpu_percent(interval=None)

        # Variabel Biomekanik Default
        sudut_lutut_kiri, sudut_lutut_kanan = 180.0, 180.0
        sudut_siku_kiri, sudut_siku_kanan = 180.0, 180.0
        sudut_pinggul_kiri = 180.0
        jarak_ankle_x, tinggi_vertikal, aspect_ratio_log = 0.0, 0.0, 0.0

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            if show_skeleton: draw_custom_skeleton(frame, landmarks)

            # Ekstraksi Biomekanik untuk kebutuhan Log CSV Skripsi
            l_hip = [landmarks[23].x, landmarks[23].y, landmarks[23].z]
            l_knee = [landmarks[25].x, landmarks[25].y, landmarks[25].z]
            l_ank = [landmarks[27].x, landmarks[27].y, landmarks[27].z]
            r_hip = [landmarks[24].x, landmarks[24].y, landmarks[24].z]
            r_knee = [landmarks[26].x, landmarks[26].y, landmarks[26].z]
            r_ank = [landmarks[28].x, landmarks[28].y, landmarks[28].z]

            sudut_lutut_kiri = calculate_angle(l_hip, l_knee, l_ank)
            sudut_lutut_kanan = calculate_angle(r_hip, r_knee, r_ank)

            l_shldr = [landmarks[11].x, landmarks[11].y, landmarks[11].z]
            l_elbow = [landmarks[13].x, landmarks[13].y, landmarks[13].z]
            l_wrist = [landmarks[15].x, landmarks[15].y, landmarks[15].z]
            r_shldr = [landmarks[12].x, landmarks[12].y, landmarks[12].z]
            r_elbow = [landmarks[14].x, landmarks[14].y, landmarks[14].z]
            r_wrist = [landmarks[16].x, landmarks[16].y, landmarks[16].z]

            sudut_siku_kiri = calculate_angle(l_shldr, l_elbow, l_wrist)
            sudut_siku_kanan = calculate_angle(r_shldr, r_elbow, r_wrist)
            sudut_pinggul_kiri = calculate_angle(l_shldr, l_hip, l_knee)
            jarak_ankle_x = abs(landmarks[27].x - landmarks[28].x)

            mid_sh_y = (landmarks[11].y + landmarks[12].y) / 2
            mid_hip_y = (landmarks[23].y + landmarks[24].y) / 2
            tinggi_vertikal = mid_hip_y - mid_sh_y

            xs = [landmarks[i].x for i in [11, 12, 23, 24, 25, 26, 27, 28]]
            ys = [landmarks[i].y for i in [11, 12, 23, 24, 25, 26, 27, 28]]
            aspect_ratio_log = (max(ys) - min(ys)) / ((max(xs) - min(xs)) + 1e-6)

            # Logika Deteksi
            if not check_visibility(landmarks):
                current_status = {'class': 'OUT_OF_FRAME', 'confidence': 0.0,
                                  'message': 'Pastikan Seluruh Tubuh Terlihat'}
                frames_queue.clear()
            else:
                try:
                    feat_45 = extract_features_45(landmarks)
                    frames_queue.append(feat_45)
                except:
                    pass

                if len(frames_queue) < SEQUENCE_LENGTH:
                    current_status = {'class': 'BUFFERING', 'confidence': 0.0, 'message': 'MENGUMPULKAN DATA...'}
                else:
                    window_data = np.array(frames_queue)

                    # Filter Gerakan (Motion Threshold)
                    if calculate_motion_score(window_data) < MOTION_THRESHOLD:
                        predicted_class, confidence = 'idle', 1.0
                    else:
                        t_start = time.time()
                        max_idx, pred_conf = ai_model.predict(window_data)
                        inference_time_ms = (time.time() - t_start) * 1000
                        raw_prediction = CLASSES_LIST[max_idx]

                        # Temporal Smoothing (Stabilisasi Prediksi)
                        PREDICTION_HISTORY.append(raw_prediction)
                        if len(PREDICTION_HISTORY) >= 9:
                            suara_olahraga = [p for p in PREDICTION_HISTORY if p != 'idle']
                            if len(suara_olahraga) >= 4:
                                predicted_class = Counter(suara_olahraga).most_common(1)[0][0]
                            else:
                                predicted_class = Counter(PREDICTION_HISTORY).most_common(1)[0][0]
                        else:
                            predicted_class = 'MENGANALISIS...'

                        confidence = pred_conf

                    if predicted_class != 'MENGANALISIS...':
                        current_status = {'class': predicted_class, 'confidence': confidence, 'message': ''}
        else:
            current_status = {'class': 'OUT_OF_FRAME', 'confidence': 0.0, 'message': 'Tidak Ada Orang!'}
            frames_queue.clear()

        # Proses Logging Data per Frame
        fps = 1 / (new_frame_time - prev_frame_time) if prev_frame_time != 0 else 0
        prev_frame_time = new_frame_time
        waktu_berjalan = frame_count / fps_kamera

        log_data = {
            "Frame": frame_count, "Waktu (Detik)": round(waktu_berjalan, 2), "FPS": round(fps, 2),
            "CPU Usage (%)": round(cpu_usage_percent, 2), "RAM Usage (MB)": round(ram_usage_mb, 2),
            "MediaPipe Time (ms)": round(mp_time_ms, 2), "AI Infer Time (ms)": round(inference_time_ms, 2),
            "Tebakan": current_status['class'], "Confidence": round(current_status['confidence'], 2),
            "Sudut_Lutut_Kiri": round(sudut_lutut_kiri, 1), "Sudut_Lutut_Kanan": round(sudut_lutut_kanan, 1),
            "Avg_Lutut": round((sudut_lutut_kiri + sudut_lutut_kanan) / 2, 1),
            "Sudut_Siku_Kiri": round(sudut_siku_kiri, 1), "Sudut_Siku_Kanan": round(sudut_siku_kanan, 1),
            "Sudut_Pinggul": round(sudut_pinggul_kiri, 1), "Jarak_Ankle_X": round(jarak_ankle_x, 3),
            "Tinggi_Vertikal": round(tinggi_vertikal, 3), "Aspect_Ratio": round(aspect_ratio_log, 3)
        }

        # Gambar UI ke layar
        frame = draw_ui_klasifikasi(frame, current_status, fps, inference_time_ms, mp_time_ms, len(frames_queue),
                                    waktu_berjalan)

        log_performa.append(log_data)
        out_video.write(frame)
        cv2.imshow(window_name, frame)

        # Kontrol Keyboard
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            show_skeleton = not show_skeleton

    # Cleanup Memory & Save Data
    cap.release()
    out_video.release()
    cv2.destroyAllWindows()

    # Simpan log dan hasilkan grafik
    if len(log_performa) > 0:
        df_log = pd.DataFrame(log_performa)
        try:
            df_log.to_csv(csv_filename, index=False)
            print(f"\nData log skripsi disimpan di: {csv_filename}")
            # Generate 4 Plot Metrik
            generate_thesis_plots(csv_filename)
        except Exception as e:
            print(f"Gagal menyimpan data: {e}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\nAplikasi dihentikan.")
        sys.exit()

EVALUASI HASIL (TRIMMED SEGMENT)

In [ ]:
import pandas as pd
import glob
import numpy as np
import os
from collections import Counter

# ==========================================
# KONFIGURASI
# ==========================================
sheet_map = {
    '0': 'Fase 1 - 0 Derajat',
    '45': 'Fase 1 - 45 Derajat',
    '90': 'Fase 1 - 90 Derajat',
    'Fase2': 'Fase 2 - Berkelanjutan'
}

WINDOW_SIZE = 30 # Ukuran jendela untuk Cara B

# Fungsi untuk menghitung ulang persentase (Diperbarui untuk handle kolom 'Subjek')
def kalkulasi_ulang_metrik(df):
    df['Accuracy (%)'] = np.where((df['TP'] + df['TN'] + df['FP'] + df['FN']) > 0,
                                  round((df['TP'] + df['TN']) / (df['TP'] + df['TN'] + df['FP'] + df['FN']) * 100, 2), 0)
    df['Precision (%)'] = np.where((df['TP'] + df['FP']) > 0,
                                   round(df['TP'] / (df['TP'] + df['FP']) * 100, 2), 0)
    df['Recall (%)'] = np.where((df['TP'] + df['FN']) > 0,
                                round(df['TP'] / (df['TP'] + df['FN']) * 100, 2), 0)
    df['F1-Score (%)'] = np.where((df['Precision (%)'] + df['Recall (%)']) > 0,
                                  round(2 * (df['Precision (%)'] * df['Recall (%)']) / (df['Precision (%)'] + df['Recall (%)']), 2), 0)
    df['Avg Confidence (%)'] = round(df['Avg Confidence (%)'], 2)

    # Deteksi apakah tabel ini level Subjek atau level Summary
    if 'Subjek' in df.columns:
        cols = ['Subjek', 'Skenario', 'Kelas Aktivitas', 'Support (Segmen)', 'TP', 'TN', 'FP', 'FN',
                'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Avg Confidence (%)']
    else:
        cols = ['Skenario', 'Kelas Aktivitas', 'Support (Segmen)', 'TP', 'TN', 'FP', 'FN',
                'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Avg Confidence (%)']
    return df[cols]

# ==========================================
# LOOP 1: EKSTRAKSI DATA MENTAH (CARA B)
# ==========================================
file_subjek = [f for f in glob.glob("*.xlsx") if not f.startswith(("Master", "Segment", "Template", "Final", "Rekap", "LASTSUBJEK", "EVALUASI"))]

print("🔍 Ekstraksi SEGMENT-LEVEL DENGAN CARA B (WINDOW 30 FRAME)...")
hasil_segmen_list = []

for file in file_subjek:
    xls = pd.ExcelFile(file)
    nama_subjek = file.replace('Data_', '').replace('.xlsx', '')

    for sheet_asli, nama_sheet_baru in sheet_map.items():
        if sheet_asli in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name=sheet_asli)

            col_gt = next((col for col in df.columns if 'ground' in str(col).lower() or 'truth' in str(col).lower()), df.columns[-1])
            col_tebakan = next((col for col in df.columns if 'tebakan' in str(col).lower() or 'pred' in str(col).lower()), df.columns[7])
            col_conf = next((col for col in df.columns if 'conf' in str(col).lower()), None)

            df['Seg_ID'] = (df[col_gt] != df[col_gt].shift()).cumsum()
            df['Is_Valid'] = df[col_tebakan].astype(str).str.upper() != 'BUFFERING'

            for seg_id, group in df.groupby('Seg_ID'):
                gerakan_asli = str(group[col_gt].iloc[0]).strip().lower()

                if pd.isna(group[col_gt].iloc[0]) or gerakan_asli == '':
                    continue

                valid_group = group[group['Is_Valid']]
                if len(valid_group) == 0: continue

                frames_pred = valid_group[col_tebakan].astype(str).str.strip().str.lower().tolist()

                window_preds = []
                for i in range(0, len(frames_pred), WINDOW_SIZE):
                    chunk = frames_pred[i : i + WINDOW_SIZE]
                    if chunk:
                        chunk_mode = Counter(chunk).most_common(1)[0][0]
                        window_preds.append(chunk_mode)

                if window_preds:
                    prediksi_mayoritas = Counter(window_preds).most_common(1)[0][0]
                else:
                    prediksi_mayoritas = "none"

                is_tp = (prediksi_mayoritas == gerakan_asli)
                avg_conf_segmen = valid_group[col_conf].mean() if col_conf else 0.0

                hasil_segmen_list.append({
                    'Subjek': nama_subjek,
                    'Skenario': nama_sheet_baru,
                    'Gerakan Asli': gerakan_asli,
                    'Prediksi Mayoritas': prediksi_mayoritas,
                    'TP_Flag': 1 if is_tp else 0,
                    'Avg_Conf': avg_conf_segmen
                })

df_hasil = pd.DataFrame(hasil_segmen_list)

# ==========================================
# LOOP 2A: HITUNG MATRIKS PER SUBJEK (DENGAN IDLE)
# ==========================================
print("📊 MENGHITUNG CONFUSION MATRIX PER SUBJEK (DENGAN IDLE)...")
metrik_list = []

for (subjek, skenario), df_sk in df_hasil.groupby(['Subjek', 'Skenario']):
    total_segmen_subjek_skenario = len(df_sk)
    kelas_unik = set(df_sk['Gerakan Asli'].unique())

    for kelas in kelas_unik:
        segmen_kelas_ini = df_sk[df_sk['Gerakan Asli'] == kelas]
        support = len(segmen_kelas_ini)

        tp = segmen_kelas_ini['TP_Flag'].sum()
        fn = support - tp
        fp = len(df_sk[(df_sk['Gerakan Asli'] != kelas) & (df_sk['Prediksi Mayoritas'] == kelas)])
        tn = total_segmen_subjek_skenario - (tp + fp + fn)

        mean_conf_kelas = segmen_kelas_ini['Avg_Conf'].mean() * 100 if support > 0 else 0.0

        metrik_list.append({
            'Subjek': subjek, 'Skenario': skenario, 'Kelas Aktivitas': kelas,
            'Support (Segmen)': support, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'Avg Confidence (%)': mean_conf_kelas
        })

df_subjek_mentah = pd.DataFrame(metrik_list)
df_subjek = kalkulasi_ulang_metrik(df_subjek_mentah.copy())
df_subjek = df_subjek.sort_values(by=['Skenario', 'Subjek', 'Kelas Aktivitas'])

# ==========================================
# LOOP 2B: HITUNG MATRIKS PER SUBJEK (TANPA IDLE) - BARU!
# ==========================================
print("📊 MENGHITUNG CONFUSION MATRIX PER SUBJEK (TANPA IDLE)...")
df_hasil_nonidle = df_hasil[df_hasil['Gerakan Asli'].str.lower() != 'idle']
metrik_list_subjek_nonidle = []

for (subjek, skenario), df_sk in df_hasil_nonidle.groupby(['Subjek', 'Skenario']):
    total_segmen = len(df_sk)
    kelas_unik = set(df_sk['Gerakan Asli'].unique())

    for kelas in kelas_unik:
        segmen_kelas_ini = df_sk[df_sk['Gerakan Asli'] == kelas]
        support = len(segmen_kelas_ini)

        tp = segmen_kelas_ini['TP_Flag'].sum()
        fn = support - tp
        fp = len(df_sk[(df_sk['Gerakan Asli'] != kelas) & (df_sk['Prediksi Mayoritas'] == kelas)])
        tn = total_segmen - (tp + fp + fn)

        avg_conf = segmen_kelas_ini['Avg_Conf'].mean() * 100 if support > 0 else 0
        metrik_list_subjek_nonidle.append({
            'Subjek': subjek, 'Skenario': skenario, 'Kelas Aktivitas': kelas,
            'Support (Segmen)': support, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'Avg Confidence (%)': avg_conf
        })

df_subjek_nonidle_mentah = pd.DataFrame(metrik_list_subjek_nonidle)
df_subjek_nonidle = kalkulasi_ulang_metrik(df_subjek_nonidle_mentah.copy())
df_subjek_nonidle = df_subjek_nonidle.sort_values(by=['Skenario', 'Subjek', 'Kelas Aktivitas'])

# ==========================================
# LOOP 3A: BUAT SHEET SUMMARY IDLE
# ==========================================
df_summary = df_subjek_mentah.groupby(['Skenario', 'Kelas Aktivitas']).agg({
    'Support (Segmen)': 'sum', 'TP': 'sum', 'TN': 'sum', 'FP': 'sum', 'FN': 'sum', 'Avg Confidence (%)': 'mean'
}).reset_index()

df_summary_idle = kalkulasi_ulang_metrik(df_summary.copy())

# ==========================================
# LOOP 3B: BUAT SHEET SUMMARY NON-IDLE (MURNI/STRICT)
# ==========================================
df_summary_nonidle_raw = df_subjek_nonidle_mentah.groupby(['Skenario', 'Kelas Aktivitas']).agg({
    'Support (Segmen)': 'sum', 'TP': 'sum', 'TN': 'sum', 'FP': 'sum', 'FN': 'sum', 'Avg Confidence (%)': 'mean'
}).reset_index()

df_summary_nonidle = kalkulasi_ulang_metrik(df_summary_nonidle_raw.copy())

# ==========================================
# EXPORT KE SATU FILE EXCEL
# ==========================================
nama_file_output = "EVALUASI_CARA_B_13SUBJEK_FINAL_LENGKAP.xlsx"

with pd.ExcelWriter(nama_file_output, engine='openpyxl') as writer:
    # Sheet 1: Detail tiap subjek tanpa Idle (Untuk lampiran skripsi)
    df_subjek_nonidle.to_excel(writer, sheet_name='Subjek_nonIdle', index=False)

    # Sheet 2: Detail tiap subjek dengan Idle (Referensi lama)
    df_subjek.to_excel(writer, sheet_name='Subjek_Idle', index=False)

    # Sheet 3: Rata-rata 13 Subjek tanpa Idle (Untuk Bab IV)
    df_summary_nonidle.to_excel(writer, sheet_name='Summary_nonIdle', index=False)

    # Sheet 4: Rata-rata 13 Subjek dengan Idle (Referensi lama)
    df_summary_idle.to_excel(writer, sheet_name='Summary_Idle', index=False)

print(f"🎉 SELESAI! File '{nama_file_output}' berhasil dibuat dengan 4 Sheet terpisah!")